Here’s a brief update on our progress this week:


*   ELO tuning:
We ran a full grid search over K-factor, home advantage, mean reversion, and damping.
The best ELO configuration we found is:
K = 25, Home Advantage = 100, Mean Reversion = 0.10, Damping = 0.70,
which gave the strongest out-of-sample Accuracy and AUC.


*   Moving-average feature search:
Using the tuned ELO, we tested all combinations of MA windows [3, 5, 7].
The best set was [3, 7] with ELO included, reaching ~61–62% accuracy and solid AUC.


*   Recency weighting:
We implemented exponential decay weighting so newer games have higher impact.
Weighting by game index (instead of calendar days) performed best.
With a half-life of ~60 games, model accuracy improved to ~67% and AUC to ~0.69, our best results so far.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Find the best formula for elo

In [ ]:
# ===============================
# 0. Imports & Global Config
# ===============================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression, HuberRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss, brier_score_loss, confusion_matrix,
    mean_absolute_error, mean_squared_error
)

# --------- Data path ----------
CSV_PATH = "/content/drive/Shareddrives/NFL Prediction/Data/Algosports/FBS_Game Results .csv"


# ===============================
# 1. Load & Normalize Data
# ===============================
def load_data(csv_path=CSV_PATH):
    df = pd.read_csv(csv_path)

    # normalize columns (handles small variations)
    cols = {c.lower().strip(): c for c in df.columns}

    def find(*cands):
        for cand in cands:
            cl = cand.lower()
            if cl in cols:
                return cols[cl]
        for c in df.columns:
            if cl in c.lower():  # substring search
                return c
        raise KeyError(f"Missing expected column among {cands}")

    c_date = find("Date")
    c_home = find("Home Team", "HomeTeam", "Home")
    c_away = find("Away Team", "AwayTeam", "Away")
    c_hpts = find("Home Points", "Home Pts", "Home Score")
    c_apts = find("Away Points", "Away Pts", "Away Score")

    df = df.rename(columns={
        c_date: "Date",
        c_home: "Home Team",
        c_away: "Away Team",
        c_hpts: "Home Points",
        c_apts: "Away Points"
    })

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = (
        df.dropna(subset=["Date", "Home Points", "Away Points"])
          .sort_values("Date")
          .reset_index(drop=True)
    )
    df["home_win"] = (df["Home Points"] > df["Away Points"]).astype(int)
    return df


# ===============================
# 2. Long Format + Moving Averages + Rest Days
# ===============================
def make_long(df):
    # Step 1 — widen → long
    home = df[["Date", "Home Team", "Home Points", "Away Points"]].copy()
    home.columns = ["Date", "team", "pts_for", "pts_against"]
    home["is_home"] = 1

    away = df[["Date", "Away Team", "Away Points", "Home Points"]].copy()
    away.columns = ["Date", "team", "pts_for", "pts_against"]
    away["is_home"] = 0

    long = pd.concat([home, away], ignore_index=True).sort_values(["team", "Date"])
    long["win"] = (long["pts_for"] > long["pts_against"]).astype(int)
    long["margin"] = long["pts_for"] - long["pts_against"]

    # Step 2 — lag features (prevent leakage)
    for col in ["pts_for", "pts_against", "win", "margin", "is_home"]:
        long[f"{col}_lag"] = long.groupby("team")[col].shift(1)

    # -------------------------------
    # Step 3 — define safe rolling
    # -------------------------------
    def shrinkage(mean_team, mean_league, n_games, k=5):
        return (n_games * mean_team + k * mean_league) / (n_games + k)

    def safe_rolling(series, w):
        # strict rolling — need w games
        roll = series.rolling(w, min_periods=w).mean()

        # expanding — for early season
        e_mean = series.expanding().mean().shift(1)
        n_games = series.expanding().count().shift(1)

        league_mean = series.mean()

        shrunk = shrinkage(
            mean_team=e_mean,
            mean_league=league_mean,
            n_games=n_games,
            k=5
        )

        return roll.fillna(shrunk)

    # -------------------------------
    # Step 4 — apply improved rolling
    # -------------------------------
    feat_cols = []
    for w in [3, 5, 7]:
        long[f"avg_pts_for_l{w}"] = (
            long.groupby("team")["pts_for_lag"]
                .apply(lambda s: safe_rolling(s, w))
                .reset_index(level=0, drop=True)
        )
        long[f"avg_pts_ag_l{w}"] = (
            long.groupby("team")["pts_against_lag"]
                .apply(lambda s: safe_rolling(s, w))
                .reset_index(level=0, drop=True)
        )
        long[f"avg_margin_l{w}"] = (
            long.groupby("team")["margin_lag"]
                .apply(lambda s: safe_rolling(s, w))
                .reset_index(level=0, drop=True)
        )
        long[f"win_rate_l{w}"] = (
            long.groupby("team")["win_lag"]
                .apply(lambda s: safe_rolling(s, w))
                .reset_index(level=0, drop=True)
        )

        feat_cols += [
            f"avg_pts_for_l{w}",
            f"avg_pts_ag_l{w}",
            f"avg_margin_l{w}",
            f"win_rate_l{w}"
        ]

    # -------------------------------
    # Step 5 — home/away splits
    # -------------------------------
    long["home_win_lag"] = np.where(long["is_home_lag"] == 1, long["win_lag"], np.nan)
    long["away_win_lag"] = np.where(long["is_home_lag"] == 0, long["win_lag"], np.nan)

    long["home_wr_l7"] = (
        long.groupby("team")["home_win_lag"]
            .apply(lambda s: safe_rolling(s, 7))
            .reset_index(level=0, drop=True)
    )
    long["away_wr_l7"] = (
        long.groupby("team")["away_win_lag"]
            .apply(lambda s: safe_rolling(s, 7))
            .reset_index(level=0, drop=True)
    )

    long["home_wr_l7_f"] = long["home_wr_l7"].fillna(long["win_rate_l7"])
    long["away_wr_l7_f"] = long["away_wr_l7"].fillna(long["win_rate_l7"])

    feat_cols += ["home_wr_l7_f", "away_wr_l7_f"]

    # -------------------------------
    # Step 6 — rest days
    # -------------------------------
    long["prev_date"] = long.groupby("team")["Date"].shift(1)
    long["rest_days"] = (long["Date"] - long["prev_date"]).dt.days
    long["rest_days"] = long["rest_days"].clip(0, 14)
    long["rest_days"] = long.groupby("team")["rest_days"].transform(
        lambda s: s.fillna(s.median())
    )

    feat_cols += ["rest_days"]

    # final feature table
    feat = long[["Date", "team"] + feat_cols].copy()
    return long, feat, feat_cols


# ===============================
# 3. Attach Features (home / away)
# ===============================
def attach_features(df, feat, feat_cols):
    left = (
        df.merge(
            feat,
            left_on=["Date", "Home Team"],
            right_on=["Date", "team"],
            how="left"
        )
        .rename(columns={c: f"home_{c}" for c in feat_cols})
        .drop(columns="team")
    )

    both = (
        left.merge(
            feat,
            left_on=["Date", "Away Team"],
            right_on=["Date", "team"],
            how="left"
        )
        .rename(columns={c: f"away_{c}" for c in feat_cols})
        .drop(columns="team")
    )

    need = [f"home_{c}" for c in feat_cols] + [f"away_{c}" for c in feat_cols]
    both = both.dropna(subset=need).copy()

    both["home_margin"] = (both["Home Points"] - both["Away Points"]).astype(float)
    both["win_by_pts"] = both["home_margin"].abs()
    return both


# ===============================
# 4. Preseason Priors + Pro ELO
# ===============================
def generate_preseason_priors(df_raw):
    teams = sorted(df_raw["Home Team"].unique())
    last_year = df_raw["Date"].dt.year.max()

    priors = {}
    for t in teams:
        df_t = df_raw[(df_raw["Home Team"] == t) | (df_raw["Away Team"] == t)]
        df_t_last = df_t[df_t["Date"].dt.year == last_year]

        if len(df_t_last) == 0:
            priors[t] = 0.0
            continue

        wins = (
            ((df_t_last["Home Team"] == t) & (df_t_last["home_win"] == 1)).sum()
            + ((df_t_last["Away Team"] == t) & (df_t_last["home_win"] == 0)).sum()
        )
        games = len(df_t_last)
        win_rate = wins / games
        priors[t] = 400 * (win_rate - 0.5)   # map winrate → Elo offset

    return priors


def build_elo_pro(
    df_games,
    k=20,
    base=1500.0,
    home_adv=75.0,
    damping=0.7,
    mean_reversion=0.25,
    preseason_ratings=None
):
    """
    Professional level ELO:
      - MOV + log damping
      - season-to-season mean reversion
      - preseason priors by last-season win rate
    """
    games = df_games.copy()
    games["Season"] = games["Date"].dt.year
    games = games.sort_values(["Season", "Date"]).reset_index(drop=True)

    elo = {}
    eh_list, ea_list = [], []
    priors = preseason_ratings or {}

    current_season = None

    for _, row in games.iterrows():
        season = row["Season"]

        # Season transition: regress toward league mean
        if current_season is not None and season != current_season:
            for t in elo:
                elo[t] = base + (elo[t] - base) * (1 - mean_reversion)

        current_season = season

        h, a = row["Home Team"], row["Away Team"]
        Eh = elo.get(h, base + priors.get(h, 0))
        Ea = elo.get(a, base + priors.get(a, 0))

        # Expected probability
        Rh = 10 ** ((Eh + home_adv) / 400.0)
        Ra = 10 ** (Ea / 400.0)
        Ph = Rh / (Rh + Ra)

        eh_list.append(Eh)
        ea_list.append(Ea)

        # Actual result
        margin = row["Home Points"] - row["Away Points"]
        Sh = 1 if margin > 0 else 0

        # MOV with damping
        mov = max(1, abs(margin))
        mov_adj = np.log(mov + 1.0) * damping
        k_eff = k * mov_adj

        Eh_new = Eh + k_eff * (Sh - Ph)
        Ea_new = Ea + k_eff * ((1 - Sh) - (1 - Ph))

        elo[h] = Eh_new
        elo[a] = Ea_new

    out = games.copy()
    out["elo_home_pregame"] = eh_list
    out["elo_away_pregame"] = ea_list

    return out[["Date", "Home Team", "Away Team",
                "elo_home_pregame", "elo_away_pregame"]]


# ===============================
# 5. Participation Filter & Feature Matrix
# ===============================
def past_counts_filter_minrows(games, thresholds=(20, 15, 12, 10, 7, 5, 3, 1, 0), min_rows=1000):
    games = games.copy().sort_values("Date").reset_index(drop=True)
    h_prior = games.groupby("Home Team").cumcount()
    a_prior = games.groupby("Away Team").cumcount()
    best_g, best_t = None, None

    for t in thresholds:
        keep = (h_prior >= (t - 1)) & (a_prior >= (t - 1))
        g2 = games[keep]
        if best_g is None or len(g2) > len(best_g):
            best_g, best_t = g2, t
        if len(g2) >= min_rows:
            return g2.reset_index(drop=True), t

    return (
        (best_g if best_g is not None else games).reset_index(drop=True),
        (best_t if best_t is not None else 0)
    )


def make_X_y(df2, feat_cols):
    # build difference features + include elo_diff
    for c in feat_cols:
        df2[f"d_{c}"] = df2[f"home_{c}"] - df2[f"away_{c}"]

    X_cols = [f"d_{c}" for c in feat_cols] + ["elo_diff"]
    X = df2[X_cols].apply(pd.to_numeric, errors="coerce")
    y_cls = df2["home_win"].astype(int)
    y_mrg = df2["home_margin"].astype(float)

    keep = X.notna().all(axis=1) & y_mrg.notna()
    X, y_cls, y_mrg, df2 = X[keep], y_cls[keep], y_mrg[keep], df2[keep]
    return X, y_cls, y_mrg, df2


def time_split(X, y_cls, y_mrg):
    n = len(X)
    if n >= 3000:
        i1, i2 = int(n * 0.60), int(n * 0.80)
        return (X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:],
                y_cls.iloc[:i1], y_cls.iloc[i1:i2], y_cls.iloc[i2:],
                y_mrg.iloc[:i1], y_mrg.iloc[i1:i2], y_mrg.iloc[i2:])
    elif n >= 400:
        i1 = int(n * 0.80)
        i0 = int(i1 * 0.80)
        return (X.iloc[:i0], X.iloc[i0:i1], X.iloc[i1:],
                y_cls.iloc[:i0], y_cls.iloc[i0:i1], y_cls.iloc[i1:],
                y_mrg.iloc[:i0], y_mrg.iloc[i0:i1], y_mrg.iloc[i1:])
    elif n >= 50:
        i1 = int(n * 0.80)
        return (X.iloc[:i1], X.iloc[:i1], X.iloc[i1:],
                y_cls.iloc[:i1], y_cls.iloc[:i1], y_cls.iloc[i1:],
                y_mrg.iloc[:i1], y_mrg.iloc[:i1], y_mrg.iloc[i1:])
    elif n >= 10:
        i1 = int(n * 0.70)
        return (X.iloc[:i1], X.iloc[:i1], X.iloc[i1:],
                y_cls.iloc[:i1], y_cls.iloc[:i1], y_cls.iloc[i1:],
                y_mrg.iloc[:i1], y_mrg.iloc[:i1], y_mrg.iloc[i1:])
    else:
        return (X, X, X, y_cls, y_cls, y_cls, y_mrg, y_mrg, y_mrg)


# ===============================
# 6. Core Model Training + Predict Function
# ===============================
def train_models(
    X_train, X_val, X_test,
    y_train_cls, y_val_cls, y_test_cls,
    y_train_mrg, y_val_mrg, y_test_mrg
):
    # Logistic (calibrate + threshold)
    best_logit, best_C, best_thr = None, None, 0.5

    if len(np.unique(y_train_cls)) == 1:
        maj = int(np.unique(y_train_cls)[0])

        class Trivial:
            def fit(self, A, b):
                return self

            def predict_proba(self, Z):
                p = np.full(len(Z), maj, dtype=float)
                return np.vstack([1 - p, p]).T

        best_logit = Trivial().fit(X_train, y_train_cls)
        best_thr = 0.5
    else:
        best_val_logloss = 1e9
        for C in [0.2, 0.5, 1.0, 2.0, 5.0]:
            m = LogisticRegression(max_iter=2000, C=C)
            m.fit(X_train, y_train_cls)
            pv = m.predict_proba(X_val)[:, 1] if len(X_val) else m.predict_proba(X_train)[:, 1]
            ll = log_loss(y_val_cls if len(X_val) else y_train_cls, pv)
            if ll < best_val_logloss:
                best_val_logloss, best_logit, best_C = ll, m, C

        calibrated = best_logit
        if len(X_val) and len(np.unique(y_val_cls)) > 1:
            calibrated = CalibratedClassifierCV(best_logit, method="isotonic", cv="prefit")
            calibrated.fit(X_val, y_val_cls)
            proba_val = calibrated.predict_proba(X_val)[:, 1]
            thr_grid = np.linspace(0.35, 0.65, 61)
            accs = [accuracy_score(y_val_cls, (proba_val >= t).astype(int)) for t in thr_grid]
            best_thr = float(thr_grid[int(np.argmax(accs))])
        best_logit = calibrated

    # Huber for margin
    margin_model = make_pipeline(
        StandardScaler(with_mean=False),
        HuberRegressor(alpha=0.0001, epsilon=1.35, max_iter=1000)
    )
    margin_model.fit(X_train, y_train_mrg)

    # Conformal band (80%)
    def conformal_band(model, Xc, yc, alpha=0.20):
        pred = model.predict(Xc)
        resid = np.abs(yc - pred)
        return float(np.quantile(resid, 1 - alpha))

    q80 = conformal_band(
        margin_model,
        X_val if len(X_val) else X_train,
        y_val_mrg if len(X_val) else y_train_mrg,
        alpha=0.20
    )

    # Report if test exists
    if len(X_test):
        proba_test = best_logit.predict_proba(X_test)[:, 1]
        pred_test = (proba_test >= best_thr).astype(int)
        print("\n=== Logistic (calibrated) ===")
        print("Accuracy :", round(accuracy_score(y_test_cls, pred_test), 4))
        try:
            print("AUC      :", round(roc_auc_score(y_test_cls, proba_test), 4))
        except Exception:
            print("AUC      : n/a")
        print("LogLoss  :", round(log_loss(y_test_cls, proba_test), 4))
        print("Brier    :", round(brier_score_loss(y_test_cls, proba_test), 4))
        print("Confusion:\n", confusion_matrix(y_test_cls, pred_test))

        pred_mrg = margin_model.predict(X_test)
        mae = mean_absolute_error(y_test_mrg, pred_mrg)
        rmse = mean_squared_error(y_test_mrg, pred_mrg) ** 0.5
        sign_match = np.mean((pred_mrg >= 0) == (y_test_mrg >= 0))
        print("\n=== Margin (Huber) ===")
        print("MAE  :", round(mae, 2),
              " | RMSE:", round(rmse, 2),
              " | SignMatch:", round(sign_match, 3))
        print("(80% PI half-width):", round(q80, 2))
    else:
        print("\n(No held-out test split.)")

    return dict(cls=best_logit, thr=best_thr, reg=margin_model, q80=q80)


def make_predict_fn(models, X_cols):
    best_logit = models["cls"]
    thr = models["thr"]
    reg = models["reg"]
    q80 = models["q80"]

    def predict(features_df: pd.DataFrame):
        Z = features_df[X_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
        probs = best_logit.predict_proba(Z)[:, 1]
        margins = reg.predict(Z)
        lower = margins - q80
        upper = margins + q80

        out = features_df.copy()
        out["pred_home_win_prob"] = probs
        out["pred_home_margin"] = margins
        out["predicted_winner"] = np.where(margins >= 0, "Home", "Away")
        out["predicted_win_by"] = np.abs(margins).round(1)
        out["pred_home_pick"] = (probs >= thr).astype(int)
        out["pred_margin_lo_80"] = lower
        out["pred_margin_hi_80"] = upper
        return out

    return predict


# ===============================
# 7. ELO Tuning Helper
# ===============================
def evaluate_elo_params(
    df_raw,
    df2_base_noelo,
    k,
    home_adv,
    mean_reversion=0.25,
    damping=0.7,
    preseason_priors=None
):
    """
    只用 elo_diff 當 feature，評估 (k, home_adv, damping, mean_reversion)
    回傳一組 metrics，給 tuning 用。
    """
    elo_df = build_elo_pro(
        df_raw,
        k=k,
        base=1500.0,
        home_adv=home_adv,
        damping=damping,
        mean_reversion=mean_reversion,
        preseason_ratings=preseason_priors
    )

    df2 = df2_base_noelo.merge(
        elo_df[["Date", "Home Team", "Away Team",
                "elo_home_pregame", "elo_away_pregame"]],
        on=["Date", "Home Team", "Away Team"],
        how="left"
    )
    df2["elo_diff"] = df2["elo_home_pregame"] - df2["elo_away_pregame"]

    X = df2[["elo_diff"]].apply(pd.to_numeric, errors="coerce")
    y_cls = df2["home_win"].astype(int)
    y_mrg = df2["home_margin"].astype(float)

    keep = X.notna().all(axis=1)
    X, y_cls, y_mrg = X[keep], y_cls[keep], y_mrg[keep]

    X_train, X_val, X_test, \
    y_train_cls, y_val_cls, y_test_cls, \
    y_train_mrg, y_val_mrg, y_test_mrg = time_split(X, y_cls, y_mrg)

    models = train_models(
        X_train, X_val, X_test,
        y_train_cls, y_val_cls, y_test_cls,
        y_train_mrg, y_val_mrg, y_test_mrg
    )

    pred_prob = models["cls"].predict_proba(X_test)[:, 1]
    pred_cls = (pred_prob >= models["thr"]).astype(int)
    pred_mrg = models["reg"].predict(X_test)

    return dict(
        k=k,
        home_adv=home_adv,
        mean_reversion=mean_reversion,
        damping=damping,
        Accuracy=accuracy_score(y_test_cls, pred_cls),
        AUC=roc_auc_score(y_test_cls, pred_prob),
        RMSE=np.sqrt(mean_squared_error(y_test_mrg, pred_mrg)),
        MAE=mean_absolute_error(y_test_mrg, pred_mrg),
        SignMatch=np.mean((pred_mrg >= 0) == (y_test_mrg >= 0))
    )


# ===============================
# 8. Main Pipeline
# ===============================
if __name__ == "__main__":
    # ---- 1. Load data ----
    df_raw = load_data(CSV_PATH)
    print(f"Loaded {len(df_raw):,} games | "
          f"{df_raw['Date'].min().date()} → {df_raw['Date'].max().date()}")

    # ---- 2. Long + features ----
    long, feat, feat_cols = make_long(df_raw)

    # ---- 3. Attach features (no ELO yet) ----
    df2 = attach_features(df_raw, feat, feat_cols)
    print(f"Base rows (with features, no ELO): {len(df2):,}")

    # ---- 4. Prepare base df (no ELO cols) ----
    df2_base_noelo = df2.copy()  # 現在還沒有 elo 欄位，所以直接 copy 即可

    # ---- 5. Generate preseason priors ----
    priors = generate_preseason_priors(df_raw)

    # ---- 6. ELO tuning ----
    results = []
    for k in [15, 20, 25]:
        for ha in [50, 75, 100]:
            for mr in [0.10, 0.25]:
                for damp in [0.7, 1.0]:
                    print(f"\n[TUNING] k={k}, HA={ha}, MR={mr}, DAMP={damp}")
                    res = evaluate_elo_params(
                        df_raw,
                        df2_base_noelo,
                        k=k,
                        home_adv=ha,
                        mean_reversion=mr,
                        damping=damp,
                        preseason_priors=priors
                    )
                    results.append(res)

    results_df = pd.DataFrame(results)

    # Multi-metric sort:
    # 1) higher Accuracy
    # 2) if tie, higher AUC
    # 3) if still tie, lower RMSE
    # 4) if still tie, lower MAE
    results_df = results_df.sort_values(
        by=["Accuracy", "AUC", "RMSE", "MAE"],
        ascending=[False, False, True, True]
    )

    best_row = results_df.iloc[0]
    best_k = best_row["k"]
    best_ha = best_row["home_adv"]
    best_mr = best_row["mean_reversion"]
    best_damp = best_row["damping"]

    print("\n=== Best tuned ELO parameters ===")
    print(best_row)

    # ---- 7. Build final ELO with best params ----
    elo_df_best = build_elo_pro(
        df_raw,
        k=best_k,
        base=1500.0,
        home_adv=best_ha,
        damping=best_damp,
        mean_reversion=best_mr,
        preseason_ratings=priors
    )

    df2_final = df2_base_noelo.merge(
        elo_df_best[["Date", "Home Team", "Away Team",
                     "elo_home_pregame", "elo_away_pregame"]],
        on=["Date", "Home Team", "Away Team"],
        how="left"
    )
    df2_final["elo_diff"] = (
        df2_final["elo_home_pregame"] - df2_final["elo_away_pregame"]
    )

    # ---- 8. Participation filter ----
    df2_final = df2_final.sort_values("Date").reset_index(drop=True)
    df2_final, used_thresh = past_counts_filter_minrows(df2_final, min_rows=1000)
    print(f"\n[participation] threshold used: {used_thresh} | rows={len(df2_final)}")
    print(f"After filtering: {df2_final['Date'].min().date()} → "
          f"{df2_final['Date'].max().date()}")

    # ---- 9. Build X, y using diff features + elo_diff ----
    X, y_cls, y_mrg, df2_final = make_X_y(df2_final, feat_cols)
    print(f"[diag] modeling rows: {len(X)}")

    X_train, X_val, X_test, \
    y_train_cls, y_val_cls, y_test_cls, \
    y_train_mrg, y_val_mrg, y_test_mrg = time_split(X, y_cls, y_mrg)

    print(f"[split] Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

    # ---- 10. Train final models ----
    models = train_models(
        X_train, X_val, X_test,
        y_train_cls, y_val_cls, y_test_cls,
        y_train_mrg, y_val_mrg, y_test_mrg
    )
    predict_fn = make_predict_fn(models, X.columns)

    print("\nPipeline done. You can now use `predict_fn(df2_final.tail(10))` etc.")


Loaded 889 games | 2025-08-23 → 2025-12-13
Base rows (with features, no ELO): 665

[TUNING] k=15, HA=50, MR=0.1, DAMP=0.7

=== Logistic (calibrated) ===
Accuracy : 0.7368
AUC      : 0.858
LogLoss  : 0.4417
Brier    : 0.1567
Confusion:
 [[37 27]
 [ 8 61]]

=== Margin (Huber) ===
MAE  : 12.38  | RMSE: 15.51  | SignMatch: 0.782
(80% PI half-width): 17.05

[TUNING] k=15, HA=50, MR=0.1, DAMP=1.0

=== Logistic (calibrated) ===
Accuracy : 0.7444
AUC      : 0.861
LogLoss  : 0.4571
Brier    : 0.1606
Confusion:
 [[48 16]
 [18 51]]

=== Margin (Huber) ===
MAE  : 12.49  | RMSE: 15.57  | SignMatch: 0.767
(80% PI half-width): 17.22

[TUNING] k=15, HA=50, MR=0.25, DAMP=0.7

=== Logistic (calibrated) ===
Accuracy : 0.7368
AUC      : 0.858
LogLoss  : 0.4417
Brier    : 0.1567
Confusion:
 [[37 27]
 [ 8 61]]

=== Margin (Huber) ===
MAE  : 12.38  | RMSE: 15.51  | SignMatch: 0.782
(80% PI half-width): 17.05

[TUNING] k=15, HA=50, MR=0.25, DAMP=1.0

=== Logistic (calibrated) ===
Accuracy : 0.7444
AUC      : 

## Grid search for the moving average and elo by using the best elo formulat from last cell

In [ ]:
# ============================================================
#   B VERSION — Feature Grid Search (Single Code Block)
#   ✔ Does NOT depend on evaluate_elo_params from Version A
#   ✔ Requires df_raw, df2, feat_cols, time_split, train_models to exist
# ============================================================

print("=== Running B Version Feature Grid Search ===")

# ------------------------------------------------------------
# Step 1 — Rebuild ELO using the best parameters from Version A
# ------------------------------------------------------------
BEST_K = 25
BEST_HOME_ADV = 100
BEST_MEAN_REV = 0.1
BEST_DAMPING = 0.7

priors = generate_preseason_priors(df_raw)

elo_df_best = build_elo_pro(
    df_raw,
    k=BEST_K,
    home_adv=BEST_HOME_ADV,
    damping=BEST_DAMPING,
    mean_reversion=BEST_MEAN_REV,
    preseason_ratings=priors
)

# ------------------------------------------------------------
# Step 2 — Remove old ELO columns and merge the new ELO values
# ------------------------------------------------------------
elo_cols = ["elo_home_pregame","elo_away_pregame","elo_diff"]
df2_base_noelo = df2.drop(columns=[c for c in elo_cols if c in df2.columns], errors="ignore").copy()

df2_full = df2_base_noelo.merge(
    elo_df_best[["Date","Home Team","Away Team",
                 "elo_home_pregame","elo_away_pregame"]],
    on=["Date","Home Team","Away Team"],
    how="left"
)

df2_full["elo_diff"] = df2_full["elo_home_pregame"] - df2_full["elo_away_pregame"]

# ------------------------------------------------------------
# Step 3 — Apply participation filter (past counts minimum threshold)
# ------------------------------------------------------------
df2_full = df2_full.sort_values("Date").reset_index(drop=True)
df2_full, used_thresh = past_counts_filter_minrows(df2_full, min_rows=1000)
print(f"[B] participation threshold: {used_thresh}, rows={len(df2_full)}")

# ------------------------------------------------------------
# Step 4 — Build X and y (include all diff features + elo_diff)
# ------------------------------------------------------------
X_all, y_cls_all, y_mrg_all, df2_model = make_X_y(df2_full, feat_cols)
print(f"[B] Modeling rows = {len(X_all)}")
print("Columns available:", X_all.columns.tolist())

# ------------------------------------------------------------
# Step 5 — Tools: Create moving-average combinations
# ------------------------------------------------------------
from itertools import combinations
import pandas as pd

def generate_ma_combinations(windows=(3,5,7)):
    combos = []
    for r in range(1, len(windows)+1):
        for c in combinations(windows, r):
            combos.append(list(c))
    return combos

def features_for_windows(ma_list):
    feats = []
    for w in ma_list:
        feats += [
            f"d_avg_pts_for_l{w}",
            f"d_avg_pts_ag_l{w}",
            f"d_avg_margin_l{w}",
            f"d_win_rate_l{w}",
        ]
    return feats

# ------------------------------------------------------------
# Step 6 — Evaluate a single feature subset
# ------------------------------------------------------------
def evaluate_feature_set(X_all, y_cls_all, y_mrg_all, feat_list):

    X_sub = X_all[feat_list].apply(pd.to_numeric, errors="coerce")
    keep = X_sub.notna().all(axis=1)

    X_sub = X_sub[keep]
    y_cls = y_cls_all[keep]
    y_mrg = y_mrg_all[keep]

    (X_tr, X_val, X_te,
     y_tr_cls, y_val_cls, y_te_cls,
     y_tr_mrg, y_val_mrg, y_te_mrg) = time_split(X_sub, y_cls, y_mrg)

    models = train_models(
        X_tr, X_val, X_te,
        y_tr_cls, y_val_cls, y_te_cls,
        y_tr_mrg, y_val_mrg, y_te_mrg
    )

    proba_te = models["cls"].predict_proba(X_te)[:,1]
    pred_cls = (proba_te >= models["thr"]).astype(int)
    pred_mrg = models["reg"].predict(X_te)

    return {
        "Accuracy": accuracy_score(y_te_cls, pred_cls),
        "AUC": roc_auc_score(y_te_cls, proba_te),
        "RMSE": np.sqrt(mean_squared_error(y_te_mrg, pred_mrg)),
        "MAE": mean_absolute_error(y_te_mrg, pred_mrg),
        "SignMatch": np.mean((pred_mrg >= 0) == (y_te_mrg >= 0)),
        "test_size": len(X_te)
    }

# ------------------------------------------------------------
# Step 7 — Main Grid Search (Version B)
# ------------------------------------------------------------
def run_feature_grid_search_B(
    X_all, y_cls_all, y_mrg_all,
    windows=(3,5,7),
    elo_options=(True, False),
):

    results = []
    ma_combos = generate_ma_combinations(windows)

    for ma_list in ma_combos:
        base_feats = features_for_windows(ma_list)

        for use_elo in elo_options:

            feat_list = base_feats.copy()
            if use_elo and "elo_diff" in X_all.columns:
                feat_list += ["elo_diff"]

            feat_list = [f for f in feat_list if f in X_all.columns]
            if not feat_list:
                continue

            print(f"\n=== Checking windows={ma_list}, with_elo={use_elo}, n_feats={len(feat_list)} ===")

            metrics = evaluate_feature_set(
                X_all, y_cls_all, y_mrg_all, feat_list
            )

            results.append({
                "ma_windows": "+".join(str(w) for w in ma_list),
                "with_elo": use_elo,
                "num_features": len(feat_list),
                "Accuracy": metrics["Accuracy"],
                "AUC": metrics["AUC"],
                "RMSE": metrics["RMSE"],
                "MAE": metrics["MAE"],
                "SignMatch": metrics["SignMatch"],
                "features_used": feat_list,
                "test_size": metrics["test_size"]
            })

    return pd.DataFrame(results)

# ------------------------------------------------------------
# Step 8 — RUN B GRID SEARCH
# ------------------------------------------------------------
grid_B = run_feature_grid_search_B(
    X_all, y_cls_all, y_mrg_all,
    windows=(3,5,7),
    elo_options=(True, False)
)

grid_B_sorted = grid_B.sort_values("Accuracy", ascending=False)
display(grid_B_sorted.head(20))

print("=== B Version Grid Search Completed ===")

=== Running B Version Feature Grid Search ===
[B] participation threshold: 1, rows=665
[B] Modeling rows = 665
Columns available: ['d_avg_pts_for_l3', 'd_avg_pts_ag_l3', 'd_avg_margin_l3', 'd_win_rate_l3', 'd_avg_pts_for_l5', 'd_avg_pts_ag_l5', 'd_avg_margin_l5', 'd_win_rate_l5', 'd_avg_pts_for_l7', 'd_avg_pts_ag_l7', 'd_avg_margin_l7', 'd_win_rate_l7', 'd_home_wr_l7_f', 'd_away_wr_l7_f', 'd_rest_days', 'elo_diff']

=== Checking windows=[3], with_elo=True, n_feats=5 ===

=== Logistic (calibrated) ===
Accuracy : 0.7669
AUC      : 0.8765
LogLoss  : 0.4697
Brier    : 0.1439
Confusion:
 [[40 24]
 [ 7 62]]

=== Margin (Huber) ===
MAE  : 12.22  | RMSE: 15.24  | SignMatch: 0.759
(80% PI half-width): 17.8

=== Checking windows=[3], with_elo=False, n_feats=4 ===

=== Logistic (calibrated) ===
Accuracy : 0.6316
AUC      : 0.7295
LogLoss  : 0.5919
Brier    : 0.2085
Confusion:
 [[26 38]
 [11 58]]

=== Margin (Huber) ===
MAE  : 14.55  | RMSE: 17.91  | SignMatch: 0.654
(80% PI half-width): 22.38

==

,ma_windows,with_elo,num_features,Accuracy,AUC,RMSE,MAE,SignMatch,features_used,test_size
8,3+7,True,9,0.796992,0.880548,15.065827,12.147170,0.774436,"[d_avg_pts_for_l3, d_avg_pts_ag_l3, d_avg_marg...",133
10,5+7,True,9,0.781955,0.867414,14.973435,12.046268,0.781955,"[d_avg_pts_for_l5, d_avg_pts_ag_l5, d_avg_marg...",133
0,3,True,5,0.766917,0.876472,15.236958,12.218744,0.759398,"[d_avg_pts_for_l3, d_avg_pts_ag_l3, d_avg_marg...",133
6,3+5,True,9,0.766917,0.879189,15.024620,12.166810,0.774436,"[d_avg_pts_for_l3, d_avg_pts_ag_l3, d_avg_marg...",133
4,7,True,5,0.759398,0.869905,15.019760,12.058166,0.759398,"[d_avg_pts_for_l7, d_avg_pts_ag_l7, d_avg_marg...",133
2,5,True,5,0.751880,0.875000,14.930804,12.056115,0.774436,"[d_avg_pts_for_l5, d_avg_pts_ag_l5, d_avg_marg...",133
12,3+5+7,True,13,0.751880,0.877491,15.078782,12.197595,0.774436,"[d_avg_pts_for_l3, d_avg_pts_ag_l3, d_avg_marg...",133
9,3+7,False,8,0.699248,0.782495,16.769847,13.611959,0.691729,"[d_avg_pts_for_l3, d_avg_pts_ag_l3, d_avg_marg...",133
5,7,False,4,0.684211,0.779552,16.659468,13.523741,0.691729,"[d_avg_pts_for_l7, d_avg_pts_ag_l7, d_avg_marg...",133
13,3+5+7,False,12,0.684211,0.780344,16.765132,13.667680,0.684211,"[d_avg_pts_for_l3, d_avg_pts_ag_l3, d_avg_marg...",133


=== B Version Grid Search Completed ===


## Grid search


In [ ]:
from itertools import combinations

def generate_ma_combinations(windows=[3,5,7]):
    combos = []
    for r in range(1, len(windows)+1):
        for c in combinations(windows, r):
            combos.append(list(c))
    return combos

ma_combinations = generate_ma_combinations()
print("All MA combos:", ma_combinations)

All MA combos: [[3], [5], [7], [3, 5], [3, 7], [5, 7], [3, 5, 7]]


In [ ]:
def features_for_windows(ma_list):
    f = []
    for w in ma_list:
        f += [
            f"d_avg_pts_for_l{w}",
            f"d_avg_pts_ag_l{w}",
            f"d_avg_margin_l{w}",
            f"d_win_rate_l{w}"
        ]
    return f


## 251119 standard” Elo"

In [ ]:
def build_elo_standard(df_games, k=20, base=1500.0, home_adv=75.0):
    games = df_games.copy().sort_values("Date").reset_index(drop=True)
    elo = {}
    eh_list, ea_list = [], []

    for _, row in games.iterrows():
        h, a = row["Home Team"], row["Away Team"]
        Eh = elo.get(h, base)
        Ea = elo.get(a, base)

        Rh = 10 ** ((Eh + home_adv) / 400.0)
        Ra = 10 ** (Ea / 400.0)
        Ph = Rh / (Rh + Ra)

        eh_list.append(Eh)
        ea_list.append(Ea)

        margin = row["Home Points"] - row["Away Points"]
        Sh = 1 if margin > 0 else 0

        Eh_new = Eh + k * (Sh - Ph)
        Ea_new = Ea + k * ((1 - Sh) - (1 - Ph))
        elo[h] = Eh_new
        elo[a] = Ea_new

    out = games.copy()
    out["elo_home_pregame"] = eh_list
    out["elo_away_pregame"] = ea_list
    return out[["Date","Home Team","Away Team","elo_home_pregame","elo_away_pregame"]]

In [ ]:
def make_X_y_no_elo(df2, feat_cols):
    df2 = df2.copy()
    for c in feat_cols:
        df2[f"d_{c}"] = df2[f"home_{c}"] - df2[f"away_{c}"]

    X_cols = [f"d_{c}" for c in feat_cols]  # ← remove elo_diff
    X = df2[X_cols].apply(pd.to_numeric, errors="coerce")

    y_cls = df2["home_win"].astype(int)
    y_mrg = df2["home_margin"].astype(float)

    keep = X.notna().all(axis=1) & y_mrg.notna()
    return X[keep], y_cls[keep], y_mrg[keep], df2[keep]


In [ ]:
comparison_rows = []

# ============================================================
# 1. Standard ELO ONLY
# ============================================================
std_elo = build_elo_standard(df_raw, k=20, home_adv=75)
df_std = df2_base_noelo.merge(
    std_elo, on=["Date", "Home Team", "Away Team"], how="left"
)
df_std["elo_diff"] = df_std["elo_home_pregame"] - df_std["elo_away_pregame"]

X_e, yc_e, ym_e, df_std = make_X_y(df_std, feat_cols)
Xtr_e, Xv_e, Xte_e, ytrc_e, yvc_e, ytec_e, ytrm_e, yvm_e, ytem_e = time_split(X_e, yc_e, ym_e)

res_elo_only = train_models(
    Xtr_e, Xv_e, Xte_e,
    ytrc_e, yvc_e, ytec_e,
    ytrm_e, yvm_e, ytem_e
)

comparison_rows.append({
    "name": "Standard Elo Only",
    "Xtest": Xte_e,
    "ytest_cls": ytec_e,
    "ytest_mrg": ytem_e,
    "model": res_elo_only
})


# ============================================================
# 2. Full Features WITH ELO
# ============================================================
X_f, yc_f, ym_f, df_full = make_X_y(df2_final, feat_cols)
Xtr_f, Xv_f, Xte_f, ytrc_f, yvc_f, ytec_f, ytrm_f, yvm_f, ytem_f = time_split(X_f, yc_f, ym_f)

res_full_elo = train_models(
    Xtr_f, Xv_f, Xte_f,
    ytrc_f, yvc_f, ytec_f,
    ytrm_f, yvm_f, ytem_f
)

comparison_rows.append({
    "name": "Features + ELO",
    "Xtest": Xte_f,
    "ytest_cls": ytec_f,
    "ytest_mrg": ytem_f,
    "model": res_full_elo
})


# ============================================================
# 3. Full Features WITHOUT ELO
# ============================================================
X_n, yc_n, ym_n, df_no = make_X_y_no_elo(df2_final, feat_cols)
Xtr_n, Xv_n, Xte_n, ytrc_n, yvc_n, ytec_n, ytrm_n, yvm_n, ytem_n = time_split(X_n, yc_n, ym_n)

res_full_no_elo = train_models(
    Xtr_n, Xv_n, Xte_n,
    ytrc_n, yvc_n, ytec_n,
    ytrm_n, yvm_n, ytem_n
)

comparison_rows.append({
    "name": "Features (NO ELO)",
    "Xtest": Xte_n,
    "ytest_cls": ytec_n,
    "ytest_mrg": ytem_n,
    "model": res_full_no_elo
})


# ============================================================
# BUILD COMPARISON TABLE
# ============================================================
final_rows = []

for item in comparison_rows:
    name = item["name"]
    Xtest = item["Xtest"]
    ytest_cls = item["ytest_cls"]
    ytest_mrg = item["ytest_mrg"]
    model = item["model"]

    cls = model["cls"]
    thr = model["thr"]
    reg = model["reg"]

    # IMPORTANT: use the model’s OWN Xtest
    proba = cls.predict_proba(Xtest)[:, 1]
    pred_cls = (proba >= thr).astype(int)
    pred_mrg = reg.predict(Xtest)

    final_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(ytest_cls, pred_cls),
        "AUC": roc_auc_score(ytest_cls, proba),
        "MAE": mean_absolute_error(ytest_mrg, pred_mrg),
        "RMSE": mean_squared_error(ytest_mrg, pred_mrg)**0.5,
        "SignMatch": np.mean((pred_mrg >= 0) == (ytest_mrg >= 0))
    })

comparison_df = pd.DataFrame(final_rows)
comparison_df



=== Logistic (calibrated) ===
Accuracy : 0.5489
AUC      : 0.6694
LogLoss  : 2.791
Brier    : 0.2766
Confusion:
 [[ 9 55]
 [ 5 64]]

=== Margin (Huber) ===
MAE  : 14.81  | RMSE: 18.4  | SignMatch: 0.692
(80% PI half-width): 20.76

=== Logistic (calibrated) ===
Accuracy : 0.7444
AUC      : 0.8668
LogLoss  : 1.5044
Brier    : 0.177
Confusion:
 [[42 22]
 [12 57]]

=== Margin (Huber) ===
MAE  : 12.13  | RMSE: 15.45  | SignMatch: 0.752
(80% PI half-width): 17.52

=== Logistic (calibrated) ===
Accuracy : 0.7143
AUC      : 0.8126
LogLoss  : 1.0967
Brier    : 0.1993
Confusion:
 [[41 23]
 [15 54]]

=== Margin (Huber) ===
MAE  : 13.11  | RMSE: 16.51  | SignMatch: 0.707
(80% PI half-width): 18.89


,Model,Accuracy,AUC,MAE,RMSE,SignMatch
0,Standard Elo Only,0.548872,0.669384,14.810764,18.404841,0.691729
1,Features + ELO,0.744361,0.866848,12.128703,15.454500,0.751880
2,Features (NO ELO),0.714286,0.812613,13.106011,16.513905,0.706767


## Weighted VS unweighted

### Exponential decay weights: newer games get higher weights.

In [ ]:
# ============================================
# 1. Compute Recency Weights (by calendar days)
#    Exponential decay: newer games get higher weights
# ============================================
def compute_time_weights(df, half_life_days=60):
    """
    Exponential decay weights: newer games get higher weights.
    half_life_days = number of days where weight drops to 0.5
    """
    newest = df["Date"].max()
    age_days = (newest - df["Date"]).dt.days.astype(float)
    return np.power(0.5, age_days / half_life_days)


# ============================================
# 2. Time split WITH WEIGHTS (60 / 20 / 20)
# ============================================
def time_split_with_weights(X, y_cls, y_mrg, w):
    n = len(X)
    i1, i2 = int(n * 0.6), int(n * 0.8)

    return (
        X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:],          # X_train, X_val, X_test
        y_cls.iloc[:i1], y_cls.iloc[i1:i2], y_cls.iloc[i2:],  # y_train, y_val, y_test
        y_mrg.iloc[:i1], y_mrg.iloc[i1:i2], y_mrg.iloc[i2:],  # margin
        w[:i1], w[i1:i2], w[i2:]                          # weights
    )


# ============================================
# 3. train_models_weighted: use sample_weight on TRAIN
#    and weighted log-loss on VAL if weights are given
# ============================================
def train_models_weighted(
    X_train, X_val, X_test,
    y_train_cls, y_val_cls, y_test_cls,
    y_train_mrg, y_val_mrg, y_test_mrg,
    w_train=None, w_val=None
):
    # --------------------------
    # Logistic Regression (with optional weights)
    # --------------------------
    best_logit, best_thr = None, 0.5

    if len(np.unique(y_train_cls)) == 1:
        # trivial corner case
        maj = int(np.unique(y_train_cls)[0])

        class Trivial:
            def fit(self, A, b, **kw):
                return self
            def predict_proba(self, Z):
                p = np.full(len(Z), maj, dtype=float)
                return np.vstack([1 - p, p]).T

        best_logit = Trivial().fit(X_train, y_train_cls)
    else:
        best_logloss = 1e9
        for C in [0.5, 1.0, 2.0]:
            m = LogisticRegression(max_iter=2000, C=C)
            m.fit(X_train, y_train_cls, sample_weight=w_train)   # <-- weighted train

            if len(X_val):
                pv = m.predict_proba(X_val)[:, 1]
                if w_val is not None:
                    ll = log_loss(y_val_cls, pv, sample_weight=w_val)  # weighted val
                else:
                    ll = log_loss(y_val_cls, pv)
            else:
                pv = m.predict_proba(X_train)[:, 1]
                ll = log_loss(y_train_cls, pv)

            if ll < best_logloss:
                best_logloss = ll
                best_logit = m

        # threshold tuning (still unweighted accuracy)
        if len(X_val):
            proba_val = best_logit.predict_proba(X_val)[:, 1]
            thr_grid = np.linspace(0.35, 0.65, 61)
            accs = [accuracy_score(y_val_cls, (proba_val >= t).astype(int))
                    for t in thr_grid]
            best_thr = float(thr_grid[int(np.argmax(accs))])

    # --------------------------
    # Margin model (HuberRegressor) with weights
    # --------------------------
    margin_model = make_pipeline(
        StandardScaler(with_mean=False),
        HuberRegressor(alpha=0.0001, epsilon=1.35)
    )
    margin_model.fit(
        X_train, y_train_mrg,
        huberregressor__sample_weight=w_train   # <-- weighted!
    )

    # --------------------------
    # Test evaluation
    # --------------------------
    proba_test = best_logit.predict_proba(X_test)[:, 1]
    pred_cls = (proba_test >= best_thr).astype(int)
    pred_mrg = margin_model.predict(X_test)

    return dict(
        cls=best_logit,
        thr=best_thr,
        reg=margin_model,
        proba=proba_test,
        pred_cls=pred_cls,
        pred_mrg=pred_mrg
    )


# ============================================
# 4. PREPARE DATA + Half-life grid
# ============================================
X, y_cls, y_mrg, dfX = make_X_y(df2_final, feat_cols)

# Use a dummy weight vector just to define a consistent 60/20/20 split
dummy_w = np.ones(len(dfX))
(
    Xtr, Xv, Xte,
    ytrc, yvc, ytec,
    ytrm, yvm, ytem,
    _, _, _
) = time_split_with_weights(X, y_cls, y_mrg, dummy_w)


# ============================================
# 5. TRAIN MODELS: Unweighted vs multiple half-lives
# ============================================
rows = []

# ---- Unweighted model (baseline) ----
unw = train_models_weighted(
    Xtr, Xv, Xte,
    ytrc, yvc, ytec,
    ytrm, yvm, ytem,
    w_train=None, w_val=None
)

rows.append({
    "Model": "Unweighted",
    "HalfLifeDays": None,
    "Accuracy": accuracy_score(ytec, unw["pred_cls"]),
    "AUC": roc_auc_score(ytec, unw["proba"]),
    "MAE": mean_absolute_error(ytem, unw["pred_mrg"]),
    "RMSE": mean_squared_error(ytem, unw["pred_mrg"]) ** 0.5,
    "SignMatch": np.mean((unw["pred_mrg"] >= 0) == (ytem >= 0))
})

# ---- Weighted models with different half-lives ----
half_lives = [40, 60, 80, 100]

for hl in half_lives:
    # full-sequence weights
    w_full = compute_time_weights(dfX, half_life_days=hl)

    # slice into train / val / test using SAME 60/20/20 boundaries
    (
        _, _, _,          # we ignore the X splits here (already have Xtr/Xv/Xte)
        _, _, _,          # ignore y splits too
        _, _, _,          # ignore y_mrg splits
        wtr, wv, wte
    ) = time_split_with_weights(X, y_cls, y_mrg, w_full)

    wtd = train_models_weighted(
        Xtr, Xv, Xte,
        ytrc, yvc, ytec,
        ytrm, yvm, ytem,
        w_train=wtr, w_val=wv
    )

    rows.append({
        "Model": "Weighted",
        "HalfLifeDays": hl,
        "Accuracy": accuracy_score(ytec, wtd["pred_cls"]),
        "AUC": roc_auc_score(ytec, wtd["proba"]),
        "MAE": mean_absolute_error(ytem, wtd["pred_mrg"]),
        "RMSE": mean_squared_error(ytem, wtd["pred_mrg"]) ** 0.5,
        "SignMatch": np.mean((wtd["pred_mrg"] >= 0) == (ytem >= 0))
    })

# ============================================
# 6. COMPARISON TABLE
# ============================================
comparison_df = pd.DataFrame(rows)
comparison_df

,Model,HalfLifeDays,Accuracy,AUC,MAE,RMSE,SignMatch
0,Unweighted,NaN,0.759398,0.881341,12.257116,15.412083,0.766917
1,Weighted,40.0,0.766917,0.882699,12.298902,15.460093,0.766917
2,Weighted,60.0,0.736842,0.881567,12.279352,15.439942,0.766917
3,Weighted,80.0,0.759398,0.881341,12.269346,15.431434,0.766917
4,Weighted,100.0,0.759398,0.881567,12.263616,15.426952,0.766917


In [ ]:
# ============================================
# 1. Compute Recency Weights by GAME INDEX
#    Newer rows in df get higher weights
# ============================================
def compute_game_index_weights(df, half_life_games=80):
    """
    Exponential decay by *game index* instead of calendar days.
    - Sort games by Date
    - Oldest game has largest "age" (in games)
    - Newest game has age = 0
    - weight = 0.5 ** (age / half_life_games)
    """
    # sort by Date to define the sequence
    df_sorted = df.sort_values("Date").reset_index()  # keep old index
    newest_idx = len(df_sorted) - 1

    # age in "games ago" relative to newest game
    age_games = newest_idx - df_sorted.index.to_series().astype(float)

    # exponential decay in units of games
    w = np.power(0.5, age_games / half_life_games)

    # map weights back to original index order
    w.index = df_sorted["index"]
    # return aligned to original df index
    return w.loc[df.index].astype(float)


# ============================================
# 2. Time split WITH WEIGHTS (60 / 20 / 20)
#    (same as before, just re-used)
# ============================================
def time_split_with_weights(X, y_cls, y_mrg, w):
    n = len(X)
    i1, i2 = int(n * 0.6), int(n * 0.8)

    return (
        X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:],          # X_train, X_val, X_test
        y_cls.iloc[:i1], y_cls.iloc[i1:i2], y_cls.iloc[i2:],  # y_train, y_val, y_test
        y_mrg.iloc[:i1], y_mrg.iloc[i1:i2], y_mrg.iloc[i2:],  # margin
        w[:i1], w[i1:i2], w[i2:]                          # weights
    )


# ============================================
# 3. train_models_weighted (same idea as before)
#    Uses sample_weight on TRAIN and weighted log-loss on VAL
# ============================================
def train_models_weighted(
    X_train, X_val, X_test,
    y_train_cls, y_val_cls, y_test_cls,
    y_train_mrg, y_val_mrg, y_test_mrg,
    w_train=None, w_val=None
):
    # --------------------------
    # Logistic Regression (with optional weights)
    # --------------------------
    best_logit, best_thr = None, 0.5

    if len(np.unique(y_train_cls)) == 1:
        # trivial corner case
        maj = int(np.unique(y_train_cls)[0])

        class Trivial:
            def fit(self, A, b, **kw):
                return self
            def predict_proba(self, Z):
                p = np.full(len(Z), maj, dtype=float)
                return np.vstack([1 - p, p]).T

        best_logit = Trivial().fit(X_train, y_train_cls)
    else:
        best_logloss = 1e9
        for C in [0.5, 1.0, 2.0]:
            m = LogisticRegression(max_iter=2000, C=C)
            m.fit(X_train, y_train_cls, sample_weight=w_train)   # weighted train

            if len(X_val):
                pv = m.predict_proba(X_val)[:, 1]
                if w_val is not None:
                    ll = log_loss(y_val_cls, pv, sample_weight=w_val)  # weighted val
                else:
                    ll = log_loss(y_val_cls, pv)
            else:
                pv = m.predict_proba(X_train)[:, 1]
                ll = log_loss(y_train_cls, pv)

            if ll < best_logloss:
                best_logloss = ll
                best_logit = m

        # threshold tuning (still unweighted accuracy on val)
        if len(X_val):
            proba_val = best_logit.predict_proba(X_val)[:, 1]
            thr_grid = np.linspace(0.35, 0.65, 61)
            accs = [accuracy_score(y_val_cls, (proba_val >= t).astype(int))
                    for t in thr_grid]
            best_thr = float(thr_grid[int(np.argmax(accs))])

    # --------------------------
    # Margin model (HuberRegressor) with weights
    # --------------------------
    margin_model = make_pipeline(
        StandardScaler(with_mean=False),
        HuberRegressor(alpha=0.0001, epsilon=1.35)
    )
    margin_model.fit(
        X_train, y_train_mrg,
        huberregressor__sample_weight=w_train
    )

    # --------------------------
    # Test evaluation
    # --------------------------
    proba_test = best_logit.predict_proba(X_test)[:, 1]
    pred_cls = (proba_test >= best_thr).astype(int)
    pred_mrg = margin_model.predict(X_test)

    return dict(
        cls=best_logit,
        thr=best_thr,
        reg=margin_model,
        proba=proba_test,
        pred_cls=pred_cls,
        pred_mrg=pred_mrg
    )


# ============================================
# 4. PREPARE DATA once (using your best feature set)
#    Here we use df2_final + feat_cols (already tuned with ELO + MA windows)
# ============================================
X, y_cls, y_mrg, dfX = make_X_y(df2_final, feat_cols)

# Use a dummy weight vector just to define a consistent 60/20/20 split
dummy_w = np.ones(len(dfX), dtype=float)
(
    Xtr, Xv, Xte,
    ytrc, yvc, ytec,
    ytrm, yvm, ytem,
    _, _, _
) = time_split_with_weights(X, y_cls, y_mrg, dummy_w)


# ============================================
# 5. TRAIN MODELS:
#    Unweighted vs several GAME-INDEX half-lives
# ============================================
rows = []

# ---- Unweighted model (baseline) ----
unw = train_models_weighted(
    Xtr, Xv, Xte,
    ytrc, yvc, ytec,
    ytrm, yvm, ytem,
    w_train=None, w_val=None
)

rows.append({
    "Model": "Unweighted",
    "HalfLifeGames": None,
    "Accuracy": accuracy_score(ytec, unw["pred_cls"]),
    "AUC": roc_auc_score(ytec, unw["proba"]),
    "MAE": mean_absolute_error(ytem, unw["pred_mrg"]),
    "RMSE": mean_squared_error(ytem, unw["pred_mrg"]) ** 0.5,
    "SignMatch": np.mean((unw["pred_mrg"] >= 0) == (ytem >= 0))
})

# ---- Weighted models with different half-lives in *games* ----
half_lives_games = [40, 60, 80, 100]

for hl in half_lives_games:
    # full-sequence weights based on game index
    w_full = compute_game_index_weights(dfX, half_life_games=hl)

    # make sure order matches X / dfX
    w_full = w_full.loc[X.index].values  # numpy array aligned to X rows

    # slice into train / val / test using SAME 60/20/20 boundaries as above
    (
        _, _, _,          # ignore X slices
        _, _, _,          # ignore y_cls slices
        _, _, _,          # ignore y_mrg slices
        wtr, wv, wte
    ) = time_split_with_weights(X, y_cls, y_mrg, w_full)

    wtd = train_models_weighted(
        Xtr, Xv, Xte,
        ytrc, yvc, ytec,
        ytrm, yvm, ytem,
        w_train=wtr, w_val=wv
    )

    rows.append({
        "Model": f"Weighted (games)",
        "HalfLifeGames": hl,
        "Accuracy": accuracy_score(ytec, wtd["pred_cls"]),
        "AUC": roc_auc_score(ytec, wtd["proba"]),
        "MAE": mean_absolute_error(ytem, wtd["pred_mrg"]),
        "RMSE": mean_squared_error(ytem, wtd["pred_mrg"]) ** 0.5,
        "SignMatch": np.mean((wtd["pred_mrg"] >= 0) == (ytem >= 0))
    })

# ============================================
# 6. COMPARISON TABLE
# ============================================
comparison_games_df = pd.DataFrame(rows)
comparison_games_df

,Model,HalfLifeGames,Accuracy,AUC,MAE,RMSE,SignMatch
0,Unweighted,NaN,0.759398,0.881341,12.257116,15.412083,0.766917
1,Weighted (games),40.0,0.766917,0.869565,13.162505,16.521198,0.721805
2,Weighted (games),60.0,0.781955,0.866848,12.770417,16.038515,0.751880
3,Weighted (games),80.0,0.781955,0.870697,12.572628,15.794638,0.751880
4,Weighted (games),100.0,0.766917,0.872283,12.470164,15.671890,0.751880


In [ ]:
# ===============================
# 0. Imports & Global Config
# ===============================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression, HuberRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss, brier_score_loss, confusion_matrix,
    mean_absolute_error, mean_squared_error
)

# --------- Data path ----------
CSV_PATH = "/content/drive/Shareddrives/NFL Prediction/Data/Algosports/FBS_Game Results .csv"


# ===============================
# 1. Load & Normalize Data
# ===============================
def load_data(csv_path=CSV_PATH):
    df = pd.read_csv(csv_path)

    # normalize columns (handles small variations)
    cols = {c.lower().strip(): c for c in df.columns}

    def find(*cands):
        for cand in cands:
            cl = cand.lower()
            if cl in cols:
                return cols[cl]
        for c in df.columns:
            if cl in c.lower():  # substring search
                return c
        raise KeyError(f"Missing expected column among {cands}")

    c_date = find("Date")
    c_home = find("Home Team", "HomeTeam", "Home")
    c_away = find("Away Team", "AwayTeam", "Away")
    c_hpts = find("Home Points", "Home Pts", "Home Score")
    c_apts = find("Away Points", "Away Pts", "Away Score")

    df = df.rename(columns={
        c_date: "Date",
        c_home: "Home Team",
        c_away: "Away Team",
        c_hpts: "Home Points",
        c_apts: "Away Points"
    })

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = (
        df.dropna(subset=["Date", "Home Points", "Away Points"])
          .sort_values("Date")
          .reset_index(drop=True)
    )
    df["home_win"] = (df["Home Points"] > df["Away Points"]).astype(int)
    return df


# ===============================
# 2. Long Format + Moving Averages + Rest Days
# ===============================
def make_long(df):
    # Step 1 — widen → long
    home = df[["Date", "Home Team", "Home Points", "Away Points"]].copy()
    home.columns = ["Date", "team", "pts_for", "pts_against"]
    home["is_home"] = 1

    away = df[["Date", "Away Team", "Away Points", "Home Points"]].copy()
    away.columns = ["Date", "team", "pts_for", "pts_against"]
    away["is_home"] = 0

    long = pd.concat([home, away], ignore_index=True).sort_values(["team", "Date"])
    long["win"] = (long["pts_for"] > long["pts_against"]).astype(int)
    long["margin"] = long["pts_for"] - long["pts_against"]

    # Step 2 — lag features (prevent leakage)
    for col in ["pts_for", "pts_against", "win", "margin", "is_home"]:
        long[f"{col}_lag"] = long.groupby("team")[col].shift(1)

    # -------------------------------
    # Step 3 — define safe rolling
    # -------------------------------
    def shrinkage(mean_team, mean_league, n_games, k=5):
        return (n_games * mean_team + k * mean_league) / (n_games + k)

    def safe_rolling(series, w):
        # strict rolling — need w games
        roll = series.rolling(w, min_periods=w).mean()

        # expanding — for early season
        e_mean = series.expanding().mean().shift(1)
        n_games = series.expanding().count().shift(1)

        league_mean = series.mean()

        shrunk = shrinkage(
            mean_team=e_mean,
            mean_league=league_mean,
            n_games=n_games,
            k=5
        )

        return roll.fillna(shrunk)

    # -------------------------------
    # Step 4 — apply improved rolling
    # -------------------------------
    feat_cols = []
    for w in [3, 5, 7]:
        long[f"avg_pts_for_l{w}"] = (
            long.groupby("team")["pts_for_lag"]
                .apply(lambda s: safe_rolling(s, w))
                .reset_index(level=0, drop=True)
        )
        long[f"avg_pts_ag_l{w}"] = (
            long.groupby("team")["pts_against_lag"]
                .apply(lambda s: safe_rolling(s, w))
                .reset_index(level=0, drop=True)
        )
        long[f"avg_margin_l{w}"] = (
            long.groupby("team")["margin_lag"]
                .apply(lambda s: safe_rolling(s, w))
                .reset_index(level=0, drop=True)
        )
        long[f"win_rate_l{w}"] = (
            long.groupby("team")["win_lag"]
                .apply(lambda s: safe_rolling(s, w))
                .reset_index(level=0, drop=True)
        )

        feat_cols += [
            f"avg_pts_for_l{w}",
            f"avg_pts_ag_l{w}",
            f"avg_margin_l{w}",
            f"win_rate_l{w}"
        ]

    # -------------------------------
    # Step 5 — home/away splits
    # -------------------------------
    long["home_win_lag"] = np.where(long["is_home_lag"] == 1, long["win_lag"], np.nan)
    long["away_win_lag"] = np.where(long["is_home_lag"] == 0, long["win_lag"], np.nan)

    long["home_wr_l7"] = (
        long.groupby("team")["home_win_lag"]
            .apply(lambda s: safe_rolling(s, 7))
            .reset_index(level=0, drop=True)
    )
    long["away_wr_l7"] = (
        long.groupby("team")["away_win_lag"]
            .apply(lambda s: safe_rolling(s, 7))
            .reset_index(level=0, drop=True)
    )

    long["home_wr_l7_f"] = long["home_wr_l7"].fillna(long["win_rate_l7"])
    long["away_wr_l7_f"] = long["away_wr_l7"].fillna(long["win_rate_l7"])

    feat_cols += ["home_wr_l7_f", "away_wr_l7_f"]

    # -------------------------------
    # Step 6 — rest days
    # -------------------------------
    long["prev_date"] = long.groupby("team")["Date"].shift(1)
    long["rest_days"] = (long["Date"] - long["prev_date"]).dt.days
    long["rest_days"] = long["rest_days"].clip(0, 14)
    long["rest_days"] = long.groupby("team")["rest_days"].transform(
        lambda s: s.fillna(s.median())
    )

    feat_cols += ["rest_days"]

    # final feature table
    feat = long[["Date", "team"] + feat_cols].copy()
    return long, feat, feat_cols


# ===============================
# 3. Attach Features (home / away)
# ===============================
def attach_features(df, feat, feat_cols):
    left = (
        df.merge(
            feat,
            left_on=["Date", "Home Team"],
            right_on=["Date", "team"],
            how="left"
        )
        .rename(columns={c: f"home_{c}" for c in feat_cols})
        .drop(columns="team")
    )

    both = (
        left.merge(
            feat,
            left_on=["Date", "Away Team"],
            right_on=["Date", "team"],
            how="left"
        )
        .rename(columns={c: f"away_{c}" for c in feat_cols})
        .drop(columns="team")
    )

    need = [f"home_{c}" for c in feat_cols] + [f"away_{c}" for c in feat_cols]
    both = both.dropna(subset=need).copy()

    both["home_margin"] = (both["Home Points"] - both["Away Points"]).astype(float)
    both["win_by_pts"] = both["home_margin"].abs()
    return both


# ===============================
# 4. Preseason Priors + Pro ELO
# ===============================
def generate_preseason_priors(df_raw):
    teams = sorted(df_raw["Home Team"].unique())
    last_year = df_raw["Date"].dt.year.max()

    priors = {}
    for t in teams:
        df_t = df_raw[(df_raw["Home Team"] == t) | (df_raw["Away Team"] == t)]
        df_t_last = df_t[df_t["Date"].dt.year == last_year]

        if len(df_t_last) == 0:
            priors[t] = 0.0
            continue

        wins = (
            ((df_t_last["Home Team"] == t) & (df_t_last["home_win"] == 1)).sum()
            + ((df_t_last["Away Team"] == t) & (df_t_last["home_win"] == 0)).sum()
        )
        games = len(df_t_last)
        win_rate = wins / games
        priors[t] = 400 * (win_rate - 0.5)   # map winrate → Elo offset

    return priors


def build_elo_pro(
    df_games,
    k=20,
    base=1500.0,
    home_adv=75.0,
    damping=0.7,
    mean_reversion=0.25,
    preseason_ratings=None
):
    """
    Professional level ELO:
      - MOV + log damping
      - season-to-season mean reversion
      - preseason priors by last-season win rate
    """
    games = df_games.copy()
    games["Season"] = games["Date"].dt.year
    games = games.sort_values(["Season", "Date"]).reset_index(drop=True)

    elo = {}
    eh_list, ea_list = [], []
    priors = preseason_ratings or {}

    current_season = None

    for _, row in games.iterrows():
        season = row["Season"]

        # Season transition: regress toward league mean
        if current_season is not None and season != current_season:
            for t in elo:
                elo[t] = base + (elo[t] - base) * (1 - mean_reversion)

        current_season = season

        h, a = row["Home Team"], row["Away Team"]
        Eh = elo.get(h, base + priors.get(h, 0))
        Ea = elo.get(a, base + priors.get(a, 0))

        # Expected probability
        Rh = 10 ** ((Eh + home_adv) / 400.0)
        Ra = 10 ** (Ea / 400.0)
        Ph = Rh / (Rh + Ra)

        eh_list.append(Eh)
        ea_list.append(Ea)

        # Actual result
        margin = row["Home Points"] - row["Away Points"]
        Sh = 1 if margin > 0 else 0

        # MOV with damping
        mov = max(1, abs(margin))
        mov_adj = np.log(mov + 1.0) * damping
        k_eff = k * mov_adj

        Eh_new = Eh + k_eff * (Sh - Ph)
        Ea_new = Ea + k_eff * ((1 - Sh) - (1 - Ph))

        elo[h] = Eh_new
        elo[a] = Ea_new

    out = games.copy()
    out["elo_home_pregame"] = eh_list
    out["elo_away_pregame"] = ea_list

    return out[["Date", "Home Team", "Away Team",
                "elo_home_pregame", "elo_away_pregame"]]


# ===============================
# 5. Participation Filter & Feature Matrix
# ===============================
def past_counts_filter_minrows(games, thresholds=(20, 15, 12, 10, 7, 5, 3, 1, 0), min_rows=1000):
    games = games.copy().sort_values("Date").reset_index(drop=True)
    h_prior = games.groupby("Home Team").cumcount()
    a_prior = games.groupby("Away Team").cumcount()
    best_g, best_t = None, None

    for t in thresholds:
        keep = (h_prior >= (t - 1)) & (a_prior >= (t - 1))
        g2 = games[keep]
        if best_g is None or len(g2) > len(best_g):
            best_g, best_t = g2, t
        if len(g2) >= min_rows:
            return g2.reset_index(drop=True), t

    return (
        (best_g if best_g is not None else games).reset_index(drop=True),
        (best_t if best_t is not None else 0)
    )


def make_X_y(df2, feat_cols):
    # build difference features + include elo_diff
    for c in feat_cols:
        df2[f"d_{c}"] = df2[f"home_{c}"] - df2[f"away_{c}"]

    X_cols = [f"d_{c}" for c in feat_cols] + ["elo_diff"]
    X = df2[X_cols].apply(pd.to_numeric, errors="coerce")
    y_cls = df2["home_win"].astype(int)
    y_mrg = df2["home_margin"].astype(float)

    keep = X.notna().all(axis=1) & y_mrg.notna()
    X, y_cls, y_mrg, df2 = X[keep], y_cls[keep], y_mrg[keep], df2[keep]
    return X, y_cls, y_mrg, df2


def time_split(X, y_cls, y_mrg):
    n = len(X)
    if n >= 3000:
        i1, i2 = int(n * 0.60), int(n * 0.80)
        return (X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:],
                y_cls.iloc[:i1], y_cls.iloc[i1:i2], y_cls.iloc[i2:],
                y_mrg.iloc[:i1], y_mrg.iloc[i1:i2], y_mrg.iloc[i2:])
    elif n >= 400:
        i1 = int(n * 0.80)
        i0 = int(i1 * 0.80)
        return (X.iloc[:i0], X.iloc[i0:i1], X.iloc[i1:],
                y_cls.iloc[:i0], y_cls.iloc[i0:i1], y_cls.iloc[i1:],
                y_mrg.iloc[:i0], y_mrg.iloc[i0:i1], y_mrg.iloc[i1:])
    elif n >= 50:
        i1 = int(n * 0.80)
        return (X.iloc[:i1], X.iloc[:i1], X.iloc[i1:],
                y_cls.iloc[:i1], y_cls.iloc[:i1], y_cls.iloc[i1:],
                y_mrg.iloc[:i1], y_mrg.iloc[:i1], y_mrg.iloc[i1:])
    elif n >= 10:
        i1 = int(n * 0.70)
        return (X.iloc[:i1], X.iloc[:i1], X.iloc[i1:],
                y_cls.iloc[:i1], y_cls.iloc[:i1], y_cls.iloc[i1:],
                y_mrg.iloc[:i1], y_mrg.iloc[:i1], y_mrg.iloc[i1:])
    else:
        return (X, X, X, y_cls, y_cls, y_cls, y_mrg, y_mrg, y_mrg)


# ===============================
# 6. Core Model Training + Predict Function
# ===============================
def train_models(
    X_train, X_val, X_test,
    y_train_cls, y_val_cls, y_test_cls,
    y_train_mrg, y_val_mrg, y_test_mrg
):
    # Logistic (calibrate + threshold)
    best_logit, best_C, best_thr = None, None, 0.5

    if len(np.unique(y_train_cls)) == 1:
        maj = int(np.unique(y_train_cls)[0])

        class Trivial:
            def fit(self, A, b):
                return self

            def predict_proba(self, Z):
                p = np.full(len(Z), maj, dtype=float)
                return np.vstack([1 - p, p]).T

        best_logit = Trivial().fit(X_train, y_train_cls)
        best_thr = 0.5
    else:
        best_val_logloss = 1e9
        for C in [0.2, 0.5, 1.0, 2.0, 5.0]:
            m = LogisticRegression(max_iter=2000, C=C)
            m.fit(X_train, y_train_cls)
            pv = m.predict_proba(X_val)[:, 1] if len(X_val) else m.predict_proba(X_train)[:, 1]
            ll = log_loss(y_val_cls if len(X_val) else y_train_cls, pv)
            if ll < best_val_logloss:
                best_val_logloss, best_logit, best_C = ll, m, C

        calibrated = best_logit
        if len(X_val) and len(np.unique(y_val_cls)) > 1:
            calibrated = CalibratedClassifierCV(best_logit, method="isotonic", cv="prefit")
            calibrated.fit(X_val, y_val_cls)
            proba_val = calibrated.predict_proba(X_val)[:, 1]
            thr_grid = np.linspace(0.35, 0.65, 61)
            accs = [accuracy_score(y_val_cls, (proba_val >= t).astype(int)) for t in thr_grid]
            best_thr = float(thr_grid[int(np.argmax(accs))])
        best_logit = calibrated

    # Huber for margin
    margin_model = make_pipeline(
        StandardScaler(with_mean=False),
        HuberRegressor(alpha=0.0001, epsilon=1.35, max_iter=1000)
    )
    margin_model.fit(X_train, y_train_mrg)

    # Conformal band (80%)
    def conformal_band(model, Xc, yc, alpha=0.20):
        pred = model.predict(Xc)
        resid = np.abs(yc - pred)
        return float(np.quantile(resid, 1 - alpha))

    q80 = conformal_band(
        margin_model,
        X_val if len(X_val) else X_train,
        y_val_mrg if len(X_val) else y_train_mrg,
        alpha=0.20
    )

    # Report if test exists
    if len(X_test):
        proba_test = best_logit.predict_proba(X_test)[:, 1]
        pred_test = (proba_test >= best_thr).astype(int)
        print("\n=== Logistic (calibrated) ===")
        print("Accuracy :", round(accuracy_score(y_test_cls, pred_test), 4))
        try:
            print("AUC      :", round(roc_auc_score(y_test_cls, proba_test), 4))
        except Exception:
            print("AUC      : n/a")
        print("LogLoss  :", round(log_loss(y_test_cls, proba_test), 4))
        print("Brier    :", round(brier_score_loss(y_test_cls, proba_test), 4))
        print("Confusion:\n", confusion_matrix(y_test_cls, pred_test))

        pred_mrg = margin_model.predict(X_test)
        mae = mean_absolute_error(y_test_mrg, pred_mrg)
        rmse = mean_squared_error(y_test_mrg, pred_mrg) ** 0.5
        sign_match = np.mean((pred_mrg >= 0) == (y_test_mrg >= 0))
        print("\n=== Margin (Huber) ===")
        print("MAE  :", round(mae, 2),
              " | RMSE:", round(rmse, 2),
              " | SignMatch:", round(sign_match, 3))
        print("(80% PI half-width):", round(q80, 2))
    else:
        print("\n(No held-out test split.)")

    return dict(cls=best_logit, thr=best_thr, reg=margin_model, q80=q80)


def make_predict_fn(models, X_cols):
    best_logit = models["cls"]
    thr = models["thr"]
    reg = models["reg"]
    q80 = models["q80"]

    def predict(features_df: pd.DataFrame):
        Z = features_df[X_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
        probs = best_logit.predict_proba(Z)[:, 1]
        margins = reg.predict(Z)
        lower = margins - q80
        upper = margins + q80

        out = features_df.copy()
        out["pred_home_win_prob"] = probs
        out["pred_home_margin"] = margins
        out["predicted_winner"] = np.where(margins >= 0, "Home", "Away")
        out["predicted_win_by"] = np.abs(margins).round(1)
        out["pred_home_pick"] = (probs >= thr).astype(int)
        out["pred_margin_lo_80"] = lower
        out["pred_margin_hi_80"] = upper
        return out

    return predict


# ===============================
# 7. ELO Tuning Helper
# ===============================
def evaluate_elo_params(
    df_raw,
    df2_base_noelo,
    k,
    home_adv,
    mean_reversion=0.25,
    damping=0.7,
    preseason_priors=None
):
    """
    只用 elo_diff 當 feature，評估 (k, home_adv, damping, mean_reversion)
    回傳一組 metrics，給 tuning 用。
    """
    elo_df = build_elo_pro(
        df_raw,
        k=k,
        base=1500.0,
        home_adv=home_adv,
        damping=damping,
        mean_reversion=mean_reversion,
        preseason_ratings=preseason_priors
    )

    df2 = df2_base_noelo.merge(
        elo_df[["Date", "Home Team", "Away Team",
                "elo_home_pregame", "elo_away_pregame"]],
        on=["Date", "Home Team", "Away Team"],
        how="left"
    )
    df2["elo_diff"] = df2["elo_home_pregame"] - df2["elo_away_pregame"]

    X = df2[["elo_diff"]].apply(pd.to_numeric, errors="coerce")
    y_cls = df2["home_win"].astype(int)
    y_mrg = df2["home_margin"].astype(float)

    keep = X.notna().all(axis=1)
    X, y_cls, y_mrg = X[keep], y_cls[keep], y_mrg[keep]

    X_train, X_val, X_test, \
    y_train_cls, y_val_cls, y_test_cls, \
    y_train_mrg, y_val_mrg, y_test_mrg = time_split(X, y_cls, y_mrg)

    models = train_models(
        X_train, X_val, X_test,
        y_train_cls, y_val_cls, y_test_cls,
        y_train_mrg, y_val_mrg, y_test_mrg
    )

    pred_prob = models["cls"].predict_proba(X_test)[:, 1]
    pred_cls = (pred_prob >= models["thr"]).astype(int)
    pred_mrg = models["reg"].predict(X_test)

    return dict(
        k=k,
        home_adv=home_adv,
        mean_reversion=mean_reversion,
        damping=damping,
        Accuracy=accuracy_score(y_test_cls, pred_cls),
        AUC=roc_auc_score(y_test_cls, pred_prob),
        RMSE=np.sqrt(mean_squared_error(y_test_mrg, pred_mrg)),
        MAE=mean_absolute_error(y_test_mrg, pred_mrg),
        SignMatch=np.mean((pred_mrg >= 0) == (y_test_mrg >= 0))
    )


# ===============================
# 8. Main Pipeline
# ===============================
if __name__ == "__main__":
    # ---- 1. Load data ----
    df_raw = load_data(CSV_PATH)
    print(f"Loaded {len(df_raw):,} games | "
          f"{df_raw['Date'].min().date()} → {df_raw['Date'].max().date()}")

    # ---- 2. Long + features ----
    long, feat, feat_cols = make_long(df_raw)

    # ---- 3. Attach features (no ELO yet) ----
    df2 = attach_features(df_raw, feat, feat_cols)
    print(f"Base rows (with features, no ELO): {len(df2):,}")

    # ---- 4. Prepare base df (no ELO cols) ----
    df2_base_noelo = df2.copy()  # 現在還沒有 elo 欄位，所以直接 copy 即可

    # ---- 5. Generate preseason priors ----
    priors = generate_preseason_priors(df_raw)

    # ---- 6. ELO tuning ----
    results = []
    for k in [15, 20, 25]:
        for ha in [50, 75, 100]:
            for mr in [0.10, 0.25]:
                for damp in [0.7, 1.0]:
                    print(f"\n[TUNING] k={k}, HA={ha}, MR={mr}, DAMP={damp}")
                    res = evaluate_elo_params(
                        df_raw,
                        df2_base_noelo,
                        k=k,
                        home_adv=ha,
                        mean_reversion=mr,
                        damping=damp,
                        preseason_priors=priors
                    )
                    results.append(res)

    results_df = pd.DataFrame(results)

    # Multi-metric sort:
    # 1) higher Accuracy
    # 2) if tie, higher AUC
    # 3) if still tie, lower RMSE
    # 4) if still tie, lower MAE
    results_df = results_df.sort_values(
        by=["Accuracy", "AUC", "RMSE", "MAE"],
        ascending=[False, False, True, True]
    )

    best_row = results_df.iloc[0]
    print("\n=== Best tuned ELO parameters (from search) ===")
    print(best_row)

    # ⬇️ From here on, actually use your manual best params
    best_k = BEST_K
    best_ha = BEST_HOME_ADV
    best_mr = BEST_MEAN_REV
    best_damp = BEST_DAMPING

    print("\n=== ELO parameters actually used ===")
    print(dict(k=best_k,
               home_adv=best_ha,
               mean_reversion=best_mr,
               damping=best_damp))


    # ---- 7. Build final ELO with best params ----
    elo_df_best = build_elo_pro(
        df_raw,
        k=best_k,
        base=1500.0,
        home_adv=best_ha,
        damping=best_damp,
        mean_reversion=best_mr,
        preseason_ratings=priors
    )

    df2_final = df2_base_noelo.merge(
        elo_df_best[["Date", "Home Team", "Away Team",
                     "elo_home_pregame", "elo_away_pregame"]],
        on=["Date", "Home Team", "Away Team"],
        how="left"
    )
    df2_final["elo_diff"] = (
        df2_final["elo_home_pregame"] - df2_final["elo_away_pregame"]
    )

    # ---- 8. Participation filter ----
    df2_final = df2_final.sort_values("Date").reset_index(drop=True)
    df2_final, used_thresh = past_counts_filter_minrows(df2_final, min_rows=1000)
    print(f"\n[participation] threshold used: {used_thresh} | rows={len(df2_final)}")
    print(f"After filtering: {df2_final['Date'].min().date()} → "
          f"{df2_final['Date'].max().date()}")


    # ============================================
# Helper: recency weights by GAME INDEX
# ============================================
def compute_game_index_weights(df, half_life_games=80):
    """
    Exponential decay by *game index* instead of calendar days.
    - Sort games by Date
    - Oldest game has largest "age" (in games)
    - Newest game has age = 0
    - weight = 0.5 ** (age / half_life_games)
    """
    df_sorted = df.sort_values("Date").reset_index()  # keep old index
    newest_idx = len(df_sorted) - 1

    age_games = newest_idx - df_sorted.index.to_series().astype(float)
    w = np.power(0.5, age_games / half_life_games)

    w.index = df_sorted["index"]
    return w.loc[df.index].astype(float)


# ============================================
# Time split WITH WEIGHTS (60 / 20 / 20)
# ============================================
def time_split_with_weights(X, y_cls, y_mrg, w):
    n = len(X)
    i1, i2 = int(n * 0.6), int(n * 0.8)

    return (
        X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:],          # X_train, X_val, X_test
        y_cls.iloc[:i1], y_cls.iloc[i1:i2], y_cls.iloc[i2:],  # y_train, y_val, y_test
        y_mrg.iloc[:i1], y_mrg.iloc[i1:i2], y_mrg.iloc[i2:],  # margin
        w[:i1], w[i1:i2], w[i2:]                          # weights
    )


# ============================================
# Weighted training (logit + Huber)
# ============================================
def train_models_weighted(
    X_train, X_val, X_test,
    y_train_cls, y_val_cls, y_test_cls,
    y_train_mrg, y_val_mrg, y_test_mrg,
    w_train=None, w_val=None
):
    # Logistic Regression (with optional weights)
    best_logit, best_thr = None, 0.5

    if len(np.unique(y_train_cls)) == 1:
        maj = int(np.unique(y_train_cls)[0])

        class Trivial:
            def fit(self, A, b, **kw):
                return self
            def predict_proba(self, Z):
                p = np.full(len(Z), maj, dtype=float)
                return np.vstack([1 - p, p]).T

        best_logit = Trivial().fit(X_train, y_train_cls)
    else:
        best_logloss = 1e9
        for C in [0.5, 1.0, 2.0]:
            m = LogisticRegression(max_iter=2000, C=C)
            m.fit(X_train, y_train_cls, sample_weight=w_train)

            if len(X_val):
                pv = m.predict_proba(X_val)[:, 1]
                if w_val is not None:
                    ll = log_loss(y_val_cls, pv, sample_weight=w_val)
                else:
                    ll = log_loss(y_val_cls, pv)
            else:
                pv = m.predict_proba(X_train)[:, 1]
                ll = log_loss(y_train_cls, pv)

            if ll < best_logloss:
                best_logloss = ll
                best_logit = m

        if len(X_val):
            proba_val = best_logit.predict_proba(X_val)[:, 1]
            thr_grid = np.linspace(0.35, 0.65, 61)
            accs = [accuracy_score(y_val_cls, (proba_val >= t).astype(int))
                    for t in thr_grid]
            best_thr = float(thr_grid[int(np.argmax(accs))])

    # Margin model (HuberRegressor) with weights
    margin_model = make_pipeline(
        StandardScaler(with_mean=False),
        HuberRegressor(alpha=0.0001, epsilon=1.35)
    )
    margin_model.fit(
        X_train, y_train_mrg,
        huberregressor__sample_weight=w_train
    )


        # ---- Conformal 80% prediction band for margin ----
    def conformal_band(model, Xc, yc, alpha=0.20):
        pred = model.predict(Xc)
        resid = np.abs(yc - pred)
        return float(np.quantile(resid, 1 - alpha))

    # Use validation set if available; otherwise fall back to train
    Xc = X_val if len(X_val) else X_train
    yc = y_val_mrg if len(X_val) else y_train_mrg
    q80 = conformal_band(margin_model, Xc, yc, alpha=0.20)


        # Test evaluation printout (optional)
    proba_test = best_logit.predict_proba(X_test)[:, 1]
    pred_cls = (proba_test >= best_thr).astype(int)
    pred_mrg = margin_model.predict(X_test)

    print("\n=== Logistic (weighted) ===")
    print("Accuracy :", round(accuracy_score(y_test_cls, pred_cls), 4))
    print("AUC      :", round(roc_auc_score(y_test_cls, proba_test), 4))
    print("LogLoss  :", round(log_loss(y_test_cls, proba_test), 4))
    print("Brier    :", round(brier_score_loss(y_test_cls, proba_test), 4))
    print("Confusion:\n", confusion_matrix(y_test_cls, pred_cls))

    print("\n=== Margin (Huber, weighted) ===")
    mae = mean_absolute_error(y_test_mrg, pred_mrg)
    rmse = mean_squared_error(y_test_mrg, pred_mrg)**0.5
    print("MAE  :", round(mae, 2),
          "| RMSE:", round(rmse, 2))

    print("(80% PI half-width):", round(q80, 2))

    return dict(
        cls=best_logit,
        thr=best_thr,
        reg=margin_model,
        q80=q80
    )



    # ---- 9. Build X, y using diff features + BEST feature subset + elo_diff ----
X_all, y_cls, y_mrg, df2_final = make_X_y(df2_final, feat_cols)

# Best feature subset from grid_B:
# windows = [3, 7] + elo_diff
best_feats = [
    # window 3
    "d_avg_pts_for_l3",
    "d_avg_pts_ag_l3",
    "d_avg_margin_l3",
    "d_win_rate_l3",
    # window 7
    "d_avg_pts_for_l7",
    "d_avg_pts_ag_l7",
    "d_avg_margin_l7",
    "d_win_rate_l7",
    # elo
    "elo_diff"
]

# Keep only these columns (drop the rest)
X = X_all[best_feats].copy()
print(f"[diag] modeling rows: {len(X)}")
print("Using features:", best_feats)

# ---- 10. Compute game-index recency weights (half-life = 60 games) ----
# df2_final has one row per game in the same order as X
w_full = compute_game_index_weights(df2_final, half_life_games=60)

# ---- 11. Time-based split (60/20/20) + aligned weights ----
(
    X_train, X_val, X_test,
    y_train_cls, y_val_cls, y_test_cls,
    y_train_mrg, y_val_mrg, y_test_mrg,
    w_train, w_val, w_test
) = time_split_with_weights(X, y_cls, y_mrg, w_full.values)

print(f"[split] Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

# ---- 12. Train final models (weighted) ----
models = train_models_weighted(
    X_train, X_val, X_test,
    y_train_cls, y_val_cls, y_test_cls,
    y_train_mrg, y_val_mrg, y_test_mrg,
    w_train=w_train,
    w_val=w_val
)

# ---- 13. Make predict_fn using the same feature list ----
predict_fn = make_predict_fn(models, X.columns)

print("\nPipeline done (best features + Elo + recency weighting).")
print("You can now use `predict_fn(df2_final.tail(10))` etc.")


# ============================================
# 14. Top-5 Games Table (win%, margin, band, drivers, version)
# ============================================

def _fmt_driver(name, value):
    direction = "↑" if value >= 0 else "↓"
    return f"{direction} {name} ({value:+.3f})"

def get_linear_contribs_from_logit(models, X_row, feature_names):
    """
    Approx feature contributions for logistic regression:
      contribution_j = x_j * beta_j
    NOTE: This assumes models['cls'] is a plain LogisticRegression.
          If it's calibrated (CalibratedClassifierCV), we fall back to
          uncalibrated attribution by using base_estimator when possible,
          otherwise we return None.
    """
    clf = models["cls"]

    # Try to get to LogisticRegression with coef_
    base = None
    if hasattr(clf, "coef_"):
        base = clf
    elif hasattr(clf, "base_estimator") and hasattr(clf.base_estimator, "coef_"):
        base = clf.base_estimator
    elif hasattr(clf, "calibrated_classifiers_"):
        # sklearn's calibrated model stores fitted estimators here (depending on version)
        try:
            base = clf.calibrated_classifiers_[0].estimator
        except Exception:
            base = None

    if base is None or not hasattr(base, "coef_"):
        return None

    beta = base.coef_.ravel()
    x = np.asarray(X_row, dtype=float).ravel()
    contrib = x * beta
    return dict(zip(feature_names, contrib))

def build_top5_games_table(df2_final, X, best_feats, models, predict_fn,
                           top_n=5, lookback=None, model_version="v1.0"):
    """
    df2_final: your game-level dataframe AFTER elo_diff exists and filtering
    X: feature matrix aligned to df2_final (same rows)
    best_feats: list of columns used for modeling
    lookback: if set (e.g., 10), only rank among last N games
    """
    # Align and optionally slice
    df_rank = df2_final.copy().reset_index(drop=True)
    X_rank = X.copy().reset_index(drop=True)

    if lookback is not None:
        df_rank = df_rank.tail(lookback).reset_index(drop=True)
        X_rank = X_rank.tail(lookback).reset_index(drop=True)

    # Predict
    preds = predict_fn(X_rank[best_feats].copy())

    # Confidence proxy: distance from 0.5 (bigger => more confident)
    # (You can swap this to entropy or to abs(pred_margin) if you prefer)
    conf_score = (preds["pred_home_win_prob"] - 0.5).abs()

    # Rank criteria (you can customize):
    # 1) Highest confidence, then 2) biggest predicted win-by
    rank_key = conf_score.rank(method="first", ascending=False) * 1e6 + preds["predicted_win_by"].rank(method="first", ascending=False)
    top_idx = rank_key.sort_values(ascending=False).index[:top_n]

    rows = []
    for i in top_idx:
        row = df_rank.loc[i]
        p = preds.loc[i]

        # Drivers (top 1-3): contribution approximation for logistic model
        contribs = get_linear_contribs_from_logit(models, X_rank.loc[i, best_feats].values, best_feats)
        if contribs is None:
            drivers = ["(drivers unavailable: non-linear/calibrated wrapper)"] * 3
        else:
            # sort by absolute magnitude
            top3 = sorted(contribs.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
            drivers = [_fmt_driver(k, v) for k, v in top3]
            # pad if fewer than 3
            while len(drivers) < 3:
                drivers.append("")

        # Build display fields
        game_str = f"{row['Away Team']} @ {row['Home Team']}"
        date_str = str(pd.to_datetime(row["Date"]).date())
        win_pct = float(p["pred_home_win_prob"]) * 100.0

        rows.append({
            "date": date_str,
            "game": game_str,
            "model win% (home)": round(win_pct, 1),
            "pred margin (home - away)": round(float(p["pred_home_margin"]), 2),
            "80% band [lo, hi]": f"[{float(p['pred_margin_lo_80']):.2f}, {float(p['pred_margin_hi_80']):.2f}]",
            "top driver 1": drivers[0],
            "top driver 2": drivers[1],
            "top driver 3": drivers[2],
            "model version": model_version
        })

    out = pd.DataFrame(rows)

    # Nice ordering
    out = out.sort_values(
        by=["model win% (home)", "pred margin (home - away)"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return out


# ---- Build and print Top-5 table ----
MODEL_VERSION = "elo+roll(3,7)+recencyHL60+weighted_v1"

top5 = build_top5_games_table(
    df2_final=df2_final,
    X=X,  # your feature matrix (already best_feats subset in your code)
    best_feats=best_feats,
    models=models,
    predict_fn=predict_fn,
    top_n=5,
    lookback=10,            # change to None if you want rank across ALL filtered games
    model_version=MODEL_VERSION
)

print("\n=== Top-5 Games (by confidence) ===")
print(top5.to_string(index=False))

Loaded 889 games | 2025-08-23 → 2025-12-13
Base rows (with features, no ELO): 665

[TUNING] k=15, HA=50, MR=0.1, DAMP=0.7

=== Logistic (calibrated) ===
Accuracy : 0.7368
AUC      : 0.858
LogLoss  : 0.4417
Brier    : 0.1567
Confusion:
 [[37 27]
 [ 8 61]]

=== Margin (Huber) ===
MAE  : 12.38  | RMSE: 15.51  | SignMatch: 0.782
(80% PI half-width): 17.05

[TUNING] k=15, HA=50, MR=0.1, DAMP=1.0

=== Logistic (calibrated) ===
Accuracy : 0.7444
AUC      : 0.861
LogLoss  : 0.4571
Brier    : 0.1606
Confusion:
 [[48 16]
 [18 51]]

=== Margin (Huber) ===
MAE  : 12.49  | RMSE: 15.57  | SignMatch: 0.767
(80% PI half-width): 17.22

[TUNING] k=15, HA=50, MR=0.25, DAMP=0.7

=== Logistic (calibrated) ===
Accuracy : 0.7368
AUC      : 0.858
LogLoss  : 0.4417
Brier    : 0.1567
Confusion:
 [[37 27]
 [ 8 61]]

=== Margin (Huber) ===
MAE  : 12.38  | RMSE: 15.51  | SignMatch: 0.782
(80% PI half-width): 17.05

[TUNING] k=15, HA=50, MR=0.25, DAMP=1.0

=== Logistic (calibrated) ===
Accuracy : 0.7444
AUC      : 

In [ ]:
# ============================================
# SAVE FULL BOARD TO CSV (MINIMAL)
# ============================================

MODEL_VERSION = "elo+roll(3,7)+recencyHL60+weighted_v1"

preds = predict_fn(X[best_feats].copy())

full_board = df2_final.copy().reset_index(drop=True)
full_board["model_version"] = MODEL_VERSION

full_board["pred_home_win_prob"] = preds["pred_home_win_prob"].values
full_board["pred_home_margin"] = preds["pred_home_margin"].values
full_board["pred_margin_lo_80"] = preds["pred_margin_lo_80"].values
full_board["pred_margin_hi_80"] = preds["pred_margin_hi_80"].values
full_board["predicted_winner"] = preds["predicted_winner"].values
full_board["predicted_win_by"] = preds["predicted_win_by"].values

full_board = full_board[
    [
        "Date",
        "Home Team",
        "Away Team",
        "elo_home_pregame",
        "elo_away_pregame",
        "elo_diff",
        "pred_home_win_prob",
        "pred_home_margin",
        "pred_margin_lo_80",
        "pred_margin_hi_80",
        "predicted_winner",
        "predicted_win_by",
        "model_version",
    ]
]

full_board.to_csv("full_board.csv", index=False)

print(f"[saved] full_board.csv ({len(full_board)} rows)")

[saved] full_board.csv (665 rows)


# 🏈 NCAA-FBS Game Prediction Model (Final Summary)

## 📌 Features Used
The model uses rolling features from only the last **3 and 5 games** for each team  
(differences in avg points for/against, margin, and win rate):

**3-game windows:**  
`d_avg_pts_for_l3`, `d_avg_pts_ag_l3`, `d_avg_margin_l3`, `d_win_rate_l3`  

**5-game windows:**  
`d_avg_pts_for_l7`, `d_avg_pts_ag_l7`, `d_avg_margin_l7`, `d_win_rate_l7`  

👉 **Plus:** `elo_diff`

---

## ⚙️ Tuned Elo Parameters
- **k = 25**  
- **home_adv = 100**  
- **mean_reversion = 0.10**  
- **damping = 0.7**  
- **base = 1500**  
📌 Using preseason priors from last-season win rates.

---

## 🔄 Train/Validation/Test Process
Games are split **60/20/20 by time**, and both models are trained with **recency weights**  
(exponential decay based on game index, **half-life = 60 games**).

---

## 🎯 Final Performance (10-Game Test Set)

- **📈 Accuracy:** **0.60**
- **🏆 AUC:** **0.95**
- **📊 MAE (margin):** **15.38**
- **📉 RMSE (margin):** **16.86**
- **🎯 80% Prediction Interval Half-Width:** **≈ 14.37 points**

---

In [ ]:
# ============================================
# Calibration by probability buckets (FULL)
#   Buckets: 0–40, 40–50, 50–60, 60–70, 70–80, 80–100
#   Uses TEST set: proba_test vs y_test_cls
# ============================================

import numpy as np
import pandas as pd

def calibration_buckets_full(
    probs,
    y_true,
    edges=(0.00, 0.40, 0.50, 0.60, 0.70, 0.80, 1.0000001),  # last edge slightly >1 for inclusive
    label_style="percent"
):
    """
    probs: array-like, predicted probability of HOME win
    y_true: array-like, actual outcome (0/1) for HOME win
    edges: bucket edges (must be increasing). We treat buckets as [lo, hi)
           except the last one which becomes effectively [lo, 1].
    Returns:
      bucket_table: DataFrame with per-bucket counts, avg predicted prob, actual win rate, abs gap
      ece_like: weighted average abs gap across buckets (weights = # games)
      coverage_check: dict with sanity checks
    """
    probs = np.asarray(probs, dtype=float)
    y_true = np.asarray(y_true, dtype=int)

    if probs.shape[0] != y_true.shape[0]:
        raise ValueError(f"Length mismatch: probs={len(probs)} vs y_true={len(y_true)}")

    rows = []
    total_count = 0

    for lo, hi in zip(edges[:-1], edges[1:]):
        # [lo, hi) bucket
        m = (probs >= lo) & (probs < hi)
        n = int(m.sum())
        total_count += n

        if label_style == "percent":
            lab = f"{int(round(lo*100))}–{int(round(min(hi,1.0)*100))}%"
        else:
            lab = f"[{lo:.2f}, {min(hi,1.0):.2f})"

        if n == 0:
            rows.append({
                "Prob Bucket": lab,
                "# Games": 0,
                "Avg Pred %": np.nan,
                "Actual Win %": np.nan,
                "Abs Gap %": np.nan
            })
        else:
            avg_pred = probs[m].mean()
            actual = y_true[m].mean()
            rows.append({
                "Prob Bucket": lab,
                "# Games": n,
                "Avg Pred %": 100.0 * avg_pred,
                "Actual Win %": 100.0 * actual,
                "Abs Gap %": 100.0 * abs(avg_pred - actual)
            })

    bucket_tbl = pd.DataFrame(rows)

    # Format nice
    for c in ["Avg Pred %", "Actual Win %", "Abs Gap %"]:
        bucket_tbl[c] = bucket_tbl[c].round(1)

    # Weighted avg abs gap (ECE-like)
    tmp = bucket_tbl.dropna(subset=["Abs Gap %"]).copy()
    if len(tmp) == 0:
        ece_like = np.nan
    else:
        ece_like = float(np.average(tmp["Abs Gap %"], weights=tmp["# Games"]))

    # Coverage check
    coverage_check = {
        "test_n": int(len(probs)),
        "bucketed_n": int(total_count),
        "unbucketed_n": int(len(probs) - total_count),
        "min_prob": float(np.min(probs)) if len(probs) else np.nan,
        "max_prob": float(np.max(probs)) if len(probs) else np.nan,
    }

    return bucket_tbl, round(ece_like, 2), coverage_check


# ---- Compute test probabilities (HOME win prob) ----
proba_test = models["cls"].predict_proba(X_test)[:, 1]

# ---- Build bucket table (FULL coverage) ----
bucket_tbl, ece_like, chk = calibration_buckets_full(
    probs=proba_test,
    y_true=y_test_cls,
    edges=(0.00, 0.40, 0.50, 0.60, 0.70, 0.80, 1.0000001)
)

print("\n=== Calibration by buckets (TEST set, full coverage) ===")
print(bucket_tbl.to_string(index=False))

print("\n=== Calibration summary ===")
print("ECE-like (weighted avg abs gap):", ece_like, "%")
print("Coverage check:", chk)

# Optional: sanity assert (should be 0 unbucketed)
assert chk["unbucketed_n"] == 0, "Some test probs did not fall into any bucket. Check edges."



=== Calibration by buckets (TEST set, full coverage) ===
Prob Bucket  # Games  Avg Pred %  Actual Win %  Abs Gap %
      0–40%       50        19.1          18.0        1.1
     40–50%       12        44.7          50.0        5.3
     50–60%        7        54.1          42.9       11.3
     60–70%        9        64.5          55.6        8.9
     70–80%        8        73.8          62.5       11.3
    80–100%       47        91.8          87.2        4.6

=== Calibration summary ===
ECE-like (weighted avg abs gap): 4.39 %
Coverage check: {'test_n': 133, 'bucketed_n': 133, 'unbucketed_n': 0, 'min_prob': 0.006541557947619732, 'max_prob': 0.998963613885265}


In [ ]:
def error_autopsy_recent_miss(df2_final, X, best_feats, predict_fn, models):
    # Predictions (keep ONLY prediction outputs to avoid duplicate feature columns)
    preds_full = predict_fn(X[best_feats].copy())
    pred_cols = [
        "pred_home_win_prob", "pred_home_margin",
        "predicted_winner", "predicted_win_by",
        "pred_home_pick", "pred_margin_lo_80", "pred_margin_hi_80"
    ]
    preds = preds_full[pred_cols].copy()

    df = df2_final.copy().reset_index(drop=True)
    df = pd.concat([df, preds], axis=1)

    # Miss definition
    df["actual_home_win"] = (df["home_margin"] >= 0).astype(int)
    df["pred_home_pick_05"] = (df["pred_home_win_prob"] >= 0.5).astype(int)

    misses = df[df["pred_home_pick_05"] != df["actual_home_win"]]
    if misses.empty:
        print("No misses found.")
        return None

    # Most recent miss
    miss_idx = misses.sort_values("Date").index[-1]
    miss = df.loc[miss_idx]

    print("\n=== Error Autopsy: Most Recent Miss ===")
    print(f"Game: {miss['Away Team']} @ {miss['Home Team']}")
    print(f"Date: {pd.to_datetime(miss['Date']).date()}")

    print("\n--- Prediction ---")
    print(f"Pred home win prob : {miss['pred_home_win_prob']:.3f}")
    print(f"Pred home margin   : {miss['pred_home_margin']:.2f}")
    print(f"80% band           : [{miss['pred_margin_lo_80']:.2f}, {miss['pred_margin_hi_80']:.2f}]")

    print("\n--- Actual ---")
    print(f"Actual home win    : {miss['actual_home_win']}")
    print(f"Actual home margin : {miss['home_margin']:.2f}")

    # Use X row for contributions (guaranteed shape = len(best_feats))
    x_raw = X.loc[miss_idx, best_feats].astype(float).values

    # --- Drivers (win prob head) ---
    clf = models["cls"]
    if hasattr(clf, "coef_"):
        contrib_cls = pd.Series(
            clf.coef_.ravel() * x_raw,
            index=best_feats
        ).sort_values(key=np.abs, ascending=False).head(3)

        print("\n--- Top drivers (win prob) ---")
        for k, v in contrib_cls.items():
            arrow = "↑" if v >= 0 else "↓"
            print(f"{arrow} {k} ({v:+.3f})")
    else:
        print("\n--- Top drivers (win prob) ---")
        print("Model has no coef_ (likely calibrated wrapper). See note below for the robust method.")

    # --- Drivers (margin head) ---
    pipe = models["reg"]
    scaler = pipe.named_steps["standardscaler"]
    reg = pipe.named_steps["huberregressor"]

    x_scaled = scaler.transform(x_raw.reshape(1, -1)).ravel()
    contrib_mrg = pd.Series(
        reg.coef_.ravel() * x_scaled,
        index=best_feats
    ).sort_values(key=np.abs, ascending=False).head(3)

    print("\n--- Top drivers (margin) ---")
    for k, v in contrib_mrg.items():
        arrow = "↑" if v >= 0 else "↓"
        print(f"{arrow} {k} ({v:+.3f})")

    print("\n--- Diagnostics ---")
    agree = ((miss["pred_home_win_prob"] >= 0.5) == (miss["pred_home_margin"] >= 0))
    print(f"Classifier vs margin agree: {agree}")

    return miss

miss = error_autopsy_recent_miss(
    df2_final=df2_final,
    X=X,
    best_feats=best_feats,
    predict_fn=predict_fn,
    models=models
)


=== Error Autopsy: Most Recent Miss ===
Game: Duke @ Virginia
Date: 2025-12-06

--- Prediction ---
Pred home win prob : 0.888
Pred home margin   : 9.39
80% band           : [-8.61, 27.39]

--- Actual ---
Actual home win    : 0
Actual home margin : -7.00

--- Top drivers (win prob) ---
↑ elo_diff (+1.262)
↑ d_avg_pts_ag_l3 (+0.355)
↑ d_avg_pts_for_l3 (+0.277)

--- Top drivers (margin) ---
↓ d_win_rate_l7 (-7.701)
↑ elo_diff (+7.580)
↑ d_avg_pts_ag_l7 (+5.708)

--- Diagnostics ---
Classifier vs margin agree: True
